In [21]:
"""
Figure 1. Independence of Tumor Immune Activity and Host Body Composition
=========================================================================
Panel A: TCGA  — GEP vs SMA scatter
Panel B: CPTAC-3 — GEP vs SMA scatter
Panel C: Per-cancer-type Spearman r barplot
Panel D: Adjusted Cox forest (GEP, SMA, SMA×GEP)

수정사항:
  - Figure 제목: Figure 1. Independence of Tumor Immune Activity and Host Body Composition
  - A, B 박스: r = 0.06, p = 0.18 (ns) → No correlation
  - A, B 축: GEP score (Tumor immune activity) / SMA z-score (Host body composition)
  - C 하단: No significant correlation across all cancer types

입력:
  TCGA_with_genes.csv
  External_with_genes.csv

출력:
  Figure1.tiff  (dpi=300)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy import stats
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR   = r"C:\Users\nsw\Desktop\body_composition"
TCGA_FILE  = os.path.join(BASE_DIR, "TCGA_with_genes.csv")
CPTAC_FILE = os.path.join(BASE_DIR, "External_with_genes.csv")
OUT_TIFF   = os.path.join(BASE_DIR, "Figure1.tiff")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드                                 ║
# ╚══════════════════════════════════════════════════════════════╝

tcga  = pd.read_csv(TCGA_FILE)
cptac = pd.read_csv(CPTAC_FILE)

print(f"TCGA  : {len(tcga)} patients")
print(f"CPTAC : {len(cptac)} patients")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    전처리                                      ║
# ╚══════════════════════════════════════════════════════════════╝

def clean_stage(s):
    if pd.isna(s): return np.nan
    s = str(s).upper()
    if 'IV'  in s: return 4
    if 'III' in s: return 3
    if 'II'  in s: return 2
    if 'I'   in s: return 1
    return np.nan

for df in [tcga, cptac]:
    df['stage_num'] = df['stage'].apply(clean_stage)
    df['sex_num']   = (df['sex'].str.upper() == 'MALE').astype(float)

def pstar(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ╔══════════════════════════════════════════════════════════════╗
# ║                    색상 팔레트                                 ║
# ╚══════════════════════════════════════════════════════════════╝

TCGA_COLORS = {
    'KIRC':'#2166AC', 'BLCA':'#4DAF4A', 'LIHC':'#FF7F00',
    'UCEC':'#E41A1C', 'STAD':'#984EA3', 'KIRP':'#A65628',
    'COAD':'#F781BF', 'READ':'#999999'
}
CPTAC_COLORS = {
    'Uterus, NOS':'#E41A1C', 'Pancreas':'#FF7F00', 'Kidney':'#2166AC'
}
coh_colors = {'TCGA':'#4C72B0', 'CPTAC-3':'#DD8452'}

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Adjusted Cox 계산                           ║
# ╚══════════════════════════════════════════════════════════════╝

cox_data = []
for name, df in [('TCGA', tcga), ('CPTAC-3', cptac)]:
    df2 = df[['OS_time','OS','GEP_score','SMA_zscore',
               'age','sex_num','stage_num']].dropna()
    df2 = df2[df2['OS_time'] > 0].copy()
    df2['SMA_x_GEP'] = df2['GEP_score'] * df2['SMA_zscore']
    covs = ['GEP_score','SMA_zscore','SMA_x_GEP','age','stage_num']
    if df2['sex_num'].std() > 0.05:
        covs.append('sex_num')
    fit_df = df2[['OS_time','OS'] + covs].copy()
    cph = CoxPHFitter()
    cph.fit(fit_df, duration_col='OS_time', event_col='OS', robust=True)
    s = cph.summary
    for var, label in [('GEP_score', 'GEP Score'),
                        ('SMA_zscore', 'SMA z-score'),
                        ('SMA_x_GEP', 'SMA × GEP')]:
        row = s.loc[var]
        cox_data.append({
            'cohort'  : name,
            'variable': label,
            'hr'      : row['exp(coef)'],
            'lo'      : row['exp(coef) lower 95%'],
            'hi'      : row['exp(coef) upper 95%'],
            'p'       : row['p'],
            'star'    : pstar(row['p'])
        })

df_cox = pd.DataFrame(cox_data)

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Spearman r 계산                             ║
# ╚══════════════════════════════════════════════════════════════╝

r_t, p_t = stats.spearmanr(tcga['GEP_score'], tcga['SMA_zscore'])
r_c, p_c = stats.spearmanr(cptac['GEP_score'], cptac['SMA_zscore'])

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃                             ║
# ╚══════════════════════════════════════════════════════════════╝

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 2, figure=fig,
              hspace=0.42, wspace=0.32,
              left=0.08, right=0.97, top=0.91, bottom=0.08)

LS, TS, TIS, PS = 11, 9.5, 12, 14

# ════════════════════════════════════════════════════════════════
# (A) TCGA: GEP vs SMA scatter
# ════════════════════════════════════════════════════════════════

ax_a = fig.add_subplot(gs[0, 0])

for ct, grp in tcga.groupby('cancer_type'):
    ax_a.scatter(grp['GEP_score'], grp['SMA_zscore'],
                 c=TCGA_COLORS.get(ct, '#888'), alpha=0.6, s=22,
                 linewidths=0, label=ct)

x, y = tcga['GEP_score'].values, tcga['SMA_zscore'].values
m, b, *_ = stats.linregress(x, y)
xr = np.linspace(x.min(), x.max(), 200)
ax_a.plot(xr, m*xr+b, color='#333', lw=1.5, ls='--', alpha=0.8)

pt = f'p = {p_t:.3f}' if p_t >= 0.001 else f'p = {p_t:.2e}'
ax_a.text(0.05, 0.95,
          f'r = {r_t:.2f}, {pt} (ns)\nNo correlation',
          transform=ax_a.transAxes, fontsize=TS, va='top',
          bbox=dict(boxstyle='round,pad=0.35', fc='#F0F0F0', ec='#999', alpha=0.95))

ax_a.set_xlabel('GEP score (Tumor immune activity)', fontsize=LS)
ax_a.set_ylabel('SMA z-score (Host body composition)', fontsize=LS)
ax_a.set_title(f'TCGA (n = {len(tcga)})', fontsize=TIS, fontweight='bold')
ax_a.tick_params(labelsize=TS)
ax_a.spines[['top','right']].set_visible(False)
lg = ax_a.legend(fontsize=7.5, frameon=True, loc='lower right',
                 markerscale=1.2, handletextpad=0.3, labelspacing=0.2)
lg.get_frame().set_edgecolor('#ccc')
ax_a.text(-0.12, 1.05, 'A', transform=ax_a.transAxes,
          fontsize=PS, fontweight='bold')

# ════════════════════════════════════════════════════════════════
# (B) CPTAC-3: GEP vs SMA scatter
# ════════════════════════════════════════════════════════════════

ax_b = fig.add_subplot(gs[0, 1])

for ct, grp in cptac.groupby('cancer_type'):
    ax_b.scatter(grp['GEP_score'], grp['SMA_zscore'],
                 c=CPTAC_COLORS.get(ct, '#888'), alpha=0.6, s=22,
                 linewidths=0, label=ct)

x2, y2 = cptac['GEP_score'].values, cptac['SMA_zscore'].values
m2, b2, *_ = stats.linregress(x2, y2)
xr2 = np.linspace(x2.min(), x2.max(), 200)
ax_b.plot(xr2, m2*xr2+b2, color='#333', lw=1.5, ls='--', alpha=0.8)

pt2 = f'p = {p_c:.3f}' if p_c >= 0.001 else f'p = {p_c:.2e}'
ax_b.text(0.05, 0.95,
          f'r = {r_c:.2f}, {pt2} (ns)\nNo correlation',
          transform=ax_b.transAxes, fontsize=TS, va='top',
          bbox=dict(boxstyle='round,pad=0.35', fc='#F0F0F0', ec='#999', alpha=0.95))

ax_b.set_xlabel('GEP score (Tumor immune activity)', fontsize=LS)
ax_b.set_ylabel('SMA z-score (Host body composition)', fontsize=LS)
ax_b.set_title(f'CPTAC-3 (n = {len(cptac)})', fontsize=TIS, fontweight='bold')
ax_b.tick_params(labelsize=TS)
ax_b.spines[['top','right']].set_visible(False)
lg2 = ax_b.legend(fontsize=8, frameon=True, loc='lower right',
                  markerscale=1.2, handletextpad=0.3, labelspacing=0.2)
lg2.get_frame().set_edgecolor('#ccc')
ax_b.text(-0.12, 1.05, 'B', transform=ax_b.transAxes,
          fontsize=PS, fontweight='bold')

# ════════════════════════════════════════════════════════════════
# (C) Per-cancer-type correlation barplot
# ════════════════════════════════════════════════════════════════

ax_c = fig.add_subplot(gs[1, 0])

records = []
for ct, grp in tcga.groupby('cancer_type'):
    if len(grp) > 5:
        r, p = stats.spearmanr(grp['GEP_score'], grp['SMA_zscore'])
        records.append({'cancer':ct, 'r':r, 'p':p, 'cohort':'TCGA', 'n':len(grp)})
for ct, grp in cptac.groupby('cancer_type'):
    if len(grp) > 5:
        short = {'Uterus, NOS':'UCEC', 'Pancreas':'PDAC', 'Kidney':'CCRCC'}.get(ct, ct)
        r, p = stats.spearmanr(grp['GEP_score'], grp['SMA_zscore'])
        records.append({'cancer':short, 'r':r, 'p':p, 'cohort':'CPTAC-3', 'n':len(grp)})

df_cor = pd.DataFrame(records).sort_values('r')
colors_bar = ['#4C72B0' if c=='TCGA' else '#DD8452' for c in df_cor['cohort']]
ax_c.barh(range(len(df_cor)), df_cor['r'],
          color=colors_bar, alpha=0.82, edgecolor='white', height=0.7)

for i, (_, row) in enumerate(df_cor.iterrows()):
    mk = '*' if row['p'] < 0.05 else 'ns'
    cm = '#c0392b' if row['p'] < 0.05 else '#888'
    xp = row['r'] + (0.01 if row['r'] >= 0 else -0.01)
    ha = 'left' if row['r'] >= 0 else 'right'
    ax_c.text(xp, i, mk, fontsize=8, va='center', ha=ha, color=cm)

ax_c.set_yticks(range(len(df_cor)))
ax_c.set_yticklabels([f"{r['cancer']} (n={r['n']})" for _, r in df_cor.iterrows()],
                     fontsize=TS)
ax_c.axvline(0, color='black', lw=0.8)
ax_c.set_xlabel("Spearman's r (GEP vs SMA)", fontsize=LS)
ax_c.set_title('Correlation by Cancer Type', fontsize=TIS, fontweight='bold')
ax_c.tick_params(axis='x', labelsize=TS)
ax_c.spines[['top','right']].set_visible(False)
ax_c.set_xlim(-0.5, 0.5)

p1 = mpatches.Patch(color='#4C72B0', label='TCGA')
p2 = mpatches.Patch(color='#DD8452', label='CPTAC-3')
lg3 = ax_c.legend(handles=[p1,p2], fontsize=8.5, frameon=True, loc='lower right')
lg3.get_frame().set_edgecolor('#ccc')

# ── 하단 한 줄 추가 ────────────────────────────────────────
ax_c.text(0.5, -0.14,
          'No significant correlation across all cancer types',
          transform=ax_c.transAxes, ha='center', va='top',
          fontsize=9.5, style='italic', color='#555',
          bbox=dict(boxstyle='round,pad=0.3', fc='#F8F8F8', ec='#ccc', alpha=0.9))

ax_c.text(-0.18, 1.05, 'C', transform=ax_c.transAxes,
          fontsize=PS, fontweight='bold')

# ════════════════════════════════════════════════════════════════
# (D) Adjusted Cox forest plot
# ════════════════════════════════════════════════════════════════

ax_d = fig.add_subplot(gs[1, 1])

vars_order = ['GEP Score', 'SMA z-score', 'SMA × GEP']
group_gap  = 1.1
coh_offset = {'TCGA': -0.14, 'CPTAC-3': 0.14}

for vi, var in enumerate(vars_order):
    base_y = vi * group_gap
    if vi > 0:
        ax_d.axhline(base_y - group_gap*0.45, color='#e0e0e0', lw=0.9, zorder=0)
    ax_d.text(0.5, base_y + 0.35, var,
              transform=ax_d.get_yaxis_transform(),
              fontsize=10.5, fontweight='bold', va='bottom', ha='center',
              color='#222')
    for coh in ['TCGA', 'CPTAC-3']:
        row = df_cox[(df_cox['variable']==var) & (df_cox['cohort']==coh)].iloc[0]
        y   = base_y + coh_offset[coh]
        col = coh_colors[coh]
        ax_d.plot([row['lo'], row['hi']], [y, y],
                  color=col, lw=2.2, alpha=0.9, zorder=2)
        ax_d.scatter(row['hr'], y, color=col, s=60, zorder=3,
                     edgecolors='white', linewidths=0.5)
        star  = row['star']
        s_col = '#c0392b' if star != 'ns' else '#888'
        s_fw  = 'bold'    if star != 'ns' else 'normal'
        ax_d.text(row['hi'] + 0.02, y, star,
                  fontsize=9, va='center', ha='left',
                  color=s_col, fontweight=s_fw, zorder=4)

ax_d.axvline(1.0, color='#444', lw=0.9, ls='--', alpha=0.7, zorder=1)
ax_d.set_xlabel('Hazard Ratio (95% CI)', fontsize=LS)
ax_d.set_title('Adjusted Cox Model\n(age, sex, stage)', fontsize=TIS, fontweight='bold')
ax_d.set_xlim(0.2, 2.6)
ax_d.set_ylim(-0.6, len(vars_order)*group_gap - 0.2)
ax_d.set_yticks([])
ax_d.spines[['top','right','left']].set_visible(False)
ax_d.tick_params(axis='x', labelsize=TS)

pt = mpatches.Patch(color='#4C72B0', label='TCGA')
pc = mpatches.Patch(color='#DD8452', label='CPTAC-3')
lg4 = ax_d.legend(handles=[pt,pc], fontsize=9, frameon=True,
                  loc='upper right', title='Cohort', title_fontsize=9)
lg4.get_frame().set_edgecolor('#ccc')

ax_d.text(0.98, 0.02,
          "*** p<0.001\n ** p<0.01\n  * p<0.05\n ns p≥0.05",
          transform=ax_d.transAxes, fontsize=7.5,
          va='bottom', ha='right', family='monospace', color='#555',
          bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#ccc', alpha=0.9))

ax_d.text(-0.12, 1.05, 'D', transform=ax_d.transAxes,
          fontsize=PS, fontweight='bold')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle(
    'Independence of Tumor Immune Activity and Host Body Composition',
    fontsize=13.5, fontweight='bold', y=0.97)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

TCGA  : 495 patients
CPTAC : 179 patients

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure1.tiff


In [1]:
"""
Figure 2. Survival Differences Across Immune–Body Subgroups
============================================================
Panel A: TCGA KM (4 groups)
Panel B: CPTAC-3 KM (4 groups)
Panel C: HR forest — vs IH_BG (adjusted Cox)

수정사항:
  - 전체 글씨 크기 키움
  - Figure 제목: Figure 2. Survival Differences Across Immune–Body Subgroups
  - A, B: ylim 0.0~1.02 (빈공간 없음), at-risk table 제거
  - A, B 범례: lower left
  - C: TCGA(파란색) 위, CPTAC-3(주황색) 아래 순서

입력:
  TCGA_with_genes.csv
  External_with_genes.csv

출력:
  Figure2.tiff  (dpi=300)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR   = r"C:\Users\nsw\Desktop\body_composition"
TCGA_FILE  = os.path.join(BASE_DIR, "TCGA_with_genes.csv")
CPTAC_FILE = os.path.join(BASE_DIR, "External_with_genes.csv")
OUT_TIFF   = os.path.join(BASE_DIR, "Figure2.tiff")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드 & 전처리                        ║
# ╚══════════════════════════════════════════════════════════════╝

tcga  = pd.read_csv(TCGA_FILE)
cptac = pd.read_csv(CPTAC_FILE)

def clean_stage(s):
    if pd.isna(s): return np.nan
    s = str(s).upper()
    if 'IV'  in s: return 4
    if 'III' in s: return 3
    if 'II'  in s: return 2
    if 'I'   in s: return 1
    return np.nan

for df in [tcga, cptac]:
    df['stage_num'] = df['stage'].apply(clean_stage)
    df['sex_num']   = (df['sex'].astype(str).str.upper() == 'MALE').astype(float)

GROUP_MAP = {
    'Group4_HighMuscle_HighImmune': 'IH_BG',
    'Group3_HighMuscle_LowImmune' : 'IL_BG',
    'Group2_LowMuscle_HighImmune' : 'IH_BP',
    'Group1_LowMuscle_LowImmune'  : 'IL_BP'
}
GROUP_ORDER  = ['IH_BG', 'IL_BG', 'IH_BP', 'IL_BP']
GROUP_LABELS = {
    'IH_BG': 'IH_BG (Immune-High / Body-Good)',
    'IL_BG': 'IL_BG (Immune-Low / Body-Good)',
    'IH_BP': 'IH_BP (Immune-High / Body-Poor)',
    'IL_BP': 'IL_BP (Immune-Low / Body-Poor)',
}
GROUP_COLORS = {
    'IH_BG': '#2166AC', 'IL_BG': '#74ADD1',
    'IH_BP': '#F46D43', 'IL_BP': '#D73027'
}

for df in [tcga, cptac]:
    df['group'] = df['combined_group'].map(GROUP_MAP)

def pstar(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Adjusted Cox (vs IH_BG)                   ║
# ╚══════════════════════════════════════════════════════════════╝

def cox_vs_ref(df, ref='IH_BG'):
    results = {}
    df2 = df[['OS_time','OS','group','age','sex_num','stage_num','cancer_type']].dropna()
    df2 = df2[df2['OS_time'] > 0].copy()
    for g in GROUP_ORDER:
        if g == ref: continue
        sub = df2[df2['group'].isin([ref, g])].copy()
        sub['is_group'] = (sub['group'] == g).astype(int)
        covs = ['OS_time','OS','is_group','age','stage_num','cancer_type']
        if sub['sex_num'].std() > 0.05:
            sub['sex_num_'] = sub['sex_num']
            covs.append('sex_num_')
        fit_df = sub[covs].dropna()
        try:
            cph = CoxPHFitter()
            cph.fit(fit_df, duration_col='OS_time', event_col='OS', strata='cancer_type')
            s = cph.summary.loc['is_group']
            results[g] = {'hr':s['exp(coef)'], 'lo':s['exp(coef) lower 95%'],
                          'hi':s['exp(coef) upper 95%'], 'p':s['p'], 'star':pstar(s['p'])}
        except:
            results[g] = {'hr':np.nan, 'lo':np.nan, 'hi':np.nan, 'p':np.nan, 'star':'na'}
    return results

print("Cox TCGA...")
tcga_p  = multivariate_logrank_test(tcga['OS_time'], tcga['group'], tcga['OS']).p_value
tcga_hr = cox_vs_ref(tcga)

print("Cox CPTAC...")
cptac_p  = multivariate_logrank_test(cptac['OS_time'], cptac['group'], cptac['OS']).p_value
cptac_hr = cox_vs_ref(cptac)

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃                             ║
# ╚══════════════════════════════════════════════════════════════╝

fig = plt.figure(figsize=(20, 8))
fig.patch.set_facecolor('white')
gs = GridSpec(1, 3, figure=fig,
              wspace=0.40, left=0.06, right=0.97,
              top=0.86, bottom=0.13,
              width_ratios=[1, 1, 0.85])

# ── 글씨 크기 (전체 키움) ──────────────────────────────────
LS, TS, TIS, PS = 14, 12, 15, 17

# ── KM helper ─────────────────────────────────────────────
def draw_km(ax, df, title, panel_label, overall_p, legend_loc='lower left'):
    df2 = df[['OS_time','OS','group']].dropna()
    df2 = df2[df2['OS_time'] > 0].copy()
    df2['OS_months'] = df2['OS_time'] / 30.44

    kmf = KaplanMeierFitter()
    for g in GROUP_ORDER:
        sub = df2[df2['group'] == g]
        kmf.fit(sub['OS_months'], sub['OS'], label=g)
        kmf.plot_survival_function(
            ax=ax, color=GROUP_COLORS[g], lw=2,
            ci_show=True, ci_alpha=0.10,
            show_censors=True,
            censor_styles={'ms': 5, 'marker': '+'})

    p_str = (f'Log-rank p = {overall_p:.2e}'
             if overall_p < 0.001 else f'Log-rank p = {overall_p:.4f}')
    ax.text(0.97, 0.97, p_str, transform=ax.transAxes,
            fontsize=11, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', alpha=0.9))

    ax.set_xlabel('Time (months)', fontsize=LS)
    ax.set_ylabel('Overall Survival Probability', fontsize=LS)
    ax.set_title(title, fontsize=TIS, fontweight='bold')
    ax.set_ylim(0.0, 1.02)   # 빈공간 없애기
    ax.set_xlim(-2, None)
    ax.tick_params(labelsize=TS)
    ax.spines[['top','right']].set_visible(False)
    ax.text(-0.10, 1.05, panel_label, transform=ax.transAxes,
            fontsize=PS, fontweight='bold')

    handles = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g])
               for g in GROUP_ORDER]
    leg = ax.legend(handles=handles, fontsize=10, frameon=True, loc=legend_loc)
    leg.get_frame().set_edgecolor('#ccc')

# ── A: TCGA ───────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
draw_km(ax_a, tcga, f'TCGA (n = {len(tcga)})', 'A', tcga_p,
        legend_loc='lower left')

# ── B: CPTAC-3 ────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
draw_km(ax_b, cptac, f'CPTAC-3 (n = {len(cptac)})', 'B', cptac_p,
        legend_loc='lower left')

# ── C: HR forest — TCGA 위(+0.14), CPTAC-3 아래(-0.14) ───
ax_c = fig.add_subplot(gs[0, 2])

groups_c   = ['IL_BG', 'IH_BP', 'IL_BP']
# TCGA offset +0.14 (위), CPTAC-3 offset -0.14 (아래)
coh_pairs  = [('TCGA',    tcga_hr,  '#4C72B0',  +0.14),
              ('CPTAC-3', cptac_hr, '#DD8452',  -0.14)]
group_gap  = 1.0

for vi, g in enumerate(groups_c):
    base_y = vi * group_gap
    if vi > 0:
        ax_c.axhline(base_y - group_gap*0.45, color='#e0e0e0', lw=0.9, zorder=0)

    ax_c.text(0.5, base_y + 0.32, f'vs IH_BG\n({g})',
              transform=ax_c.get_yaxis_transform(),
              fontsize=11.5, fontweight='bold', va='bottom', ha='center',
              color=GROUP_COLORS[g])

    for coh, hr_dict, col, offset in coh_pairs:
        row = hr_dict[g]
        y   = base_y + offset
        if not np.isnan(row['hr']):
            ax_c.plot([row['lo'], row['hi']], [y, y],
                      color=col, lw=2.4, alpha=0.9, zorder=2)
            ax_c.scatter(row['hr'], y, color=col, s=70, zorder=3,
                         edgecolors='white', linewidths=0.5)
            star  = row['star']
            s_col = '#c0392b' if star != 'ns' else '#888'
            s_fw  = 'bold' if star != 'ns' else 'normal'
            ax_c.text(row['hi'] + 0.05, y, star,
                      fontsize=11.5, va='center', ha='left',
                      color=s_col, fontweight=s_fw, zorder=4)

ax_c.axvline(1.0, color='#444', lw=0.9, ls='--', alpha=0.7, zorder=1)
ax_c.set_xlabel('Hazard Ratio (95% CI)\nvs IH_BG (reference)', fontsize=LS)
ax_c.set_title('Adjusted HR Summary\n(age, sex, stage | strat: cancer type)',
               fontsize=12.5, fontweight='bold')
ax_c.set_xlim(0.2, 6.0)
ax_c.set_ylim(-0.55, len(groups_c)*group_gap - 0.2)
ax_c.set_yticks([])
ax_c.spines[['top','right','left']].set_visible(False)
ax_c.tick_params(axis='x', labelsize=TS)

# 범례: TCGA 먼저
pt       = mpatches.Patch(color='#4C72B0', label='TCGA')
pc_patch = mpatches.Patch(color='#DD8452', label='CPTAC-3')
lg = ax_c.legend(handles=[pt, pc_patch], fontsize=11, frameon=True,
                 loc='upper right', title='Cohort', title_fontsize=11)
lg.get_frame().set_edgecolor('#ccc')

ax_c.text(0.98, 0.02,
          "*** p<0.001\n ** p<0.01\n  * p<0.05\n ns p≥0.05",
          transform=ax_c.transAxes, fontsize=10,
          va='bottom', ha='right', family='monospace', color='#555',
          bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#ccc', alpha=0.9))

ax_c.text(-0.10, 1.05, 'C', transform=ax_c.transAxes,
          fontsize=PS, fontweight='bold')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle('Survival Differences Across Immune–Body Subgroups',
             fontsize=17, fontweight='bold', y=1.01)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Cox TCGA...
Cox CPTAC...

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure2.tiff


In [2]:
"""
Figure 3. Selection of PC6 as the Key Prognostic Latent Feature
================================================================
Panel A: Forest plot  — HR ± 95%CI, TCGA + CPTAC-3
Panel B: -log10(p) scatter — bilateral significance
(Panel C 제거)

수정사항: 전체 글씨 크기 키움

입력:  TCGA_with_genes.csv, External_with_genes.csv
       ae_latent_pca10_all.csv, ae_latent_pca10_external.csv
출력:  Figure3.tiff  (dpi=300)
실행:  python figure3.py
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# CONFIG
# ============================================================
BASE      = r"C:\Users\nsw\Desktop\body_composition"
TCGA_PATH = os.path.join(BASE, "TCGA_with_genes.csv")
EXT_PATH  = os.path.join(BASE, "External_with_genes.csv")
AE_TCGA   = os.path.join(BASE, "ae_latent_pca10_all.csv")
AE_EXT    = os.path.join(BASE, "ae_latent_pca10_external.csv")
OUTPATH   = os.path.join(BASE, "Figure3.tiff")


# ============================================================
# Helpers
# ============================================================
def star(p):
    if pd.isna(p): return 'na'
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'ns'

def safe_neglog10(p):
    if pd.isna(p): return 0.0
    return -np.log10(max(float(p), 1e-300))

def same_direction(hr1, hr2):
    if pd.isna(hr1) or pd.isna(hr2): return False
    return ((hr1 > 1) and (hr2 > 1)) or ((hr1 < 1) and (hr2 < 1))

def fmt_hr(hr, lo, hi, p):
    if any(pd.isna(x) for x in [hr, lo, hi, p]): return "NA"
    return f"HR={hr:.3f} ({lo:.3f}–{hi:.3f})  {star(p)}"

def choose_selected_pc(screen_df):
    df = screen_df.copy()
    df['t_sig']       = df['t_p'] < 0.05
    df['e_sig']       = df['e_p'] < 0.05
    df['same_dir']    = [same_direction(t_hr, e_hr)
                         for t_hr, e_hr in zip(df['t_hr'], df['e_hr'])]
    df['bilateral']   = df['t_sig'] & df['e_sig'] & df['same_dir']
    df['score']       = df['t_p'].apply(safe_neglog10) + df['e_p'].apply(safe_neglog10)
    df['effect_score']= np.abs(df['t_hr'] - 1) + np.abs(df['e_hr'] - 1)
    bilateral_df = df[df['bilateral']].copy()
    if len(bilateral_df) == 0:
        return None, df
    bilateral_df = bilateral_df.sort_values(
        by=['score', 'effect_score'], ascending=[False, False]
    ).reset_index(drop=True)
    return bilateral_df.loc[0, 'PC'], df


# ============================================================
# Load & prep
# ============================================================
print("Loading data...")
t    = pd.read_csv(TCGA_PATH)
e    = pd.read_csv(EXT_PATH)
ae_t = pd.read_csv(AE_TCGA)
ae_e = pd.read_csv(AE_EXT)

for df in [t, e]:
    df['age']       = pd.to_numeric(df['age'], errors='coerce')
    df['OS_time']   = pd.to_numeric(df['OS_time'], errors='coerce') / 30.44
    df['sex_bin']   = (df['sex'].astype(str).str.upper() == 'MALE').astype(float)
    df['stage_bin'] = df['stage'].apply(
        lambda s: 1.0 if any(x in str(s) for x in ['III', 'IV'])
        else (0.0 if any(x in str(s) for x in ['I', 'II']) else np.nan))

t['pid']    = t['patient_id'].astype(str).str[:12]
ae_t['pid'] = ae_t['patient_id'].astype(str).str[:12]
e['pid']    = e['patient_id'].astype(str)
ae_e['pid'] = ae_e['patient_id'].astype(str)

for df_clin, ae_df, name in [(t, ae_t, 't'), (e, ae_e, 'e')]:
    merged = df_clin.merge(
        ae_df[['pid'] + [f'AE_PC{i}' for i in range(1, 11)]],
        on='pid', how='inner')
    for i in range(1, 11):
        col = f'AE_PC{i}'
        std = merged[col].std()
        merged[f'{col}_z'] = (
            (merged[col] - merged[col].mean()) / std
            if (not pd.isna(std) and std > 0) else np.nan)
    if name == 't': df_t = merged
    else:           df_e = merged

print(f"  TCGA: {len(df_t)}  |  CPTAC-3: {len(df_e)}")


# ============================================================
# Multivariable Cox: each PC
# ============================================================
print("Computing Cox...")
rows = []

for i in range(1, 11):
    pc_col = f'AE_PC{i}_z'
    row = {'PC': f'PC{i}'}
    for tag, df_ in [('t', df_t), ('e', df_e)]:
        d = df_[['OS_time', 'OS', 'cancer_type', 'age', 'sex_bin',
                  'stage_bin', 'SMA_zscore', pc_col]].dropna()
        d = d[d['OS_time'] > 0]
        try:
            cph = CoxPHFitter(penalizer=0.1)
            cph.fit(d, duration_col='OS_time', event_col='OS', strata='cancer_type')
            r = cph.summary.loc[pc_col]
            row[f'{tag}_hr'] = r['exp(coef)']
            row[f'{tag}_lo'] = r['exp(coef) lower 95%']
            row[f'{tag}_hi'] = r['exp(coef) upper 95%']
            row[f'{tag}_p']  = r['p']
        except Exception:
            row[f'{tag}_hr'] = np.nan; row[f'{tag}_lo'] = np.nan
            row[f'{tag}_hi'] = np.nan; row[f'{tag}_p']  = 1.0
    print(f"  PC{i}: TCGA HR={row['t_hr']:.3f} p={row['t_p']:.4f} | "
          f"CPTAC-3 HR={row['e_hr']:.3f} p={row['e_p']:.4f}")
    rows.append(row)

SCREEN = pd.DataFrame(rows)
selected_pc, SCREEN = choose_selected_pc(SCREEN)

selected_idx = None
sel_row      = None
if selected_pc:
    selected_idx   = SCREEN.index[SCREEN['PC'] == selected_pc][0]
    sel_row        = SCREEN[SCREEN['PC'] == selected_pc].iloc[0]
    direction_text = "same direction" if sel_row['same_dir'] else "discordant direction"
else:
    direction_text = "no replicated signal"

SIG = -np.log10(0.05)
print(f"\nSelected PC: {selected_pc}")


# ============================================================
# Figure  (1 row × 2 panels — C 제거)
# ============================================================
fig = plt.figure(figsize=(22, 10))
fig.patch.set_facecolor('white')

gs = GridSpec(1, 2, figure=fig,
              wspace=0.48, left=0.07, right=0.97,
              top=0.86, bottom=0.10,
              width_ratios=[1.2, 1.0])


# ──────────────────────────────────────────────────────────────
# Panel A: Forest plot
# ──────────────────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
n = len(SCREEN); ys = np.arange(n)[::-1]; OFF = 0.20
ax_a.axvline(1, color='#888', lw=1.2, ls='--', zorder=0)

for i, row in SCREEN.iterrows():
    y      = ys[i]
    is_sel = bool(selected_pc and row['PC'] == selected_pc)
    t_s    = row['t_sig']; e_s = row['e_sig']

    if is_sel:
        ax_a.axhspan(y-0.46, y+0.46, color='#FADBD8', alpha=0.45, zorder=1)
    elif i % 2 == 0:
        ax_a.axhspan(y-0.46, y+0.46, color='#F8F9FA', alpha=0.4,  zorder=0)

    # TCGA
    tc = '#2980B9' if t_s else '#AED6F1'
    ax_a.errorbar(row['t_hr'], y+OFF,
                  xerr=[[row['t_hr']-row['t_lo']], [row['t_hi']-row['t_hr']]],
                  fmt='D' if t_s else 'o', color=tc,
                  markersize=12 if (t_s and is_sel) else (10 if t_s else 6),
                  elinewidth=2.2 if t_s else 1.0, capsize=4, zorder=4,
                  markeredgecolor='#111' if t_s else tc,
                  markeredgewidth=0.9 if t_s else 0)

    # CPTAC-3
    ec = '#E74C3C' if e_s else '#F1948A'
    ax_a.errorbar(row['e_hr'], y-OFF,
                  xerr=[[row['e_hr']-row['e_lo']], [row['e_hi']-row['e_hr']]],
                  fmt='s' if e_s else 'o', color=ec,
                  markersize=12 if (e_s and is_sel) else (10 if e_s else 6),
                  elinewidth=2.2 if e_s else 1.0, capsize=4, zorder=4,
                  markeredgecolor='#111' if e_s else ec,
                  markeredgewidth=0.9 if e_s else 0)

    ax_a.text(-0.02, y, row['PC'], va='center', ha='right', fontsize=13,
              fontweight='bold' if is_sel else 'normal',
              color='#C0392B' if is_sel else '#2C3E50',
              transform=ax_a.get_yaxis_transform())
    ax_a.text(2.90, y+OFF,
              fmt_hr(row['t_hr'], row['t_lo'], row['t_hi'], row['t_p']),
              va='center', ha='left', fontsize=10.5,
              color='#2980B9' if t_s else '#CCC',
              fontweight='bold' if t_s else 'normal')
    ax_a.text(2.90, y-OFF,
              fmt_hr(row['e_hr'], row['e_lo'], row['e_hi'], row['e_p']),
              va='center', ha='left', fontsize=10.5,
              color='#E74C3C' if e_s else '#CCC',
              fontweight='bold' if e_s else 'normal')
    if is_sel:
        for yo, co in [(OFF, '#2980B9'), (-OFF, '#E74C3C')]:
            ax_a.text(0.03, y+yo, '★', va='center', ha='left',
                      fontsize=13, color=co, zorder=5,
                      transform=ax_a.get_yaxis_transform())

ax_a.set_xlim(0.45, 2.85); ax_a.set_ylim(-0.8, n-0.15); ax_a.set_yticks([])
ax_a.set_xlabel('Hazard Ratio  (95% CI)', fontsize=14)
ax_a.tick_params(axis='x', labelsize=12)
ax_a.set_title('PC1–PC10 Multivariable Cox Forest Plot\n'
               'adj: SMA, age, sex, stage  |  strat: cancer type  |  OS',
               fontsize=14, fontweight='bold')
ax_a.text(2.88, n-0.42, 'HR (95% CI)   sig',
          va='center', ha='left', fontsize=11, fontweight='bold', color='#555')

leg_a = [
    Line2D([0],[0], marker='D', color='w', markerfacecolor='#2980B9',
           markeredgecolor='#111', markersize=11, label='TCGA  p<0.05'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#E74C3C',
           markeredgecolor='#111', markersize=11, label='CPTAC-3  p<0.05'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#BDC3C7',
           markersize=8, label='ns'),
]
if selected_pc:
    leg_a.append(mpatches.Patch(facecolor='#FADBD8', alpha=0.6,
                                edgecolor='#C0392B', label=f'{selected_pc}  (selected)'))
ax_a.legend(handles=leg_a, fontsize=11, framealpha=0.9, loc='lower right')
for sp in ['top', 'right', 'left']: ax_a.spines[sp].set_visible(False)
ax_a.text(-0.06, 1.05, 'A', transform=ax_a.transAxes, fontsize=17, fontweight='bold')


# ──────────────────────────────────────────────────────────────
# Panel B: Bilateral significance scatter
# ──────────────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
t_lp = [safe_neglog10(p) for p in SCREEN['t_p']]
e_lp = [safe_neglog10(p) for p in SCREEN['e_p']]
eff  = np.abs(SCREEN['t_hr'].values-1) + np.abs(SCREEN['e_hr'].values-1)
eff  = np.nan_to_num(eff, nan=0.0)
sz   = eff * 700 + 60

xmax = max(max(t_lp)*1.5, SIG+0.8, 2.0)
ymax = max(max(e_lp)*1.9, SIG+0.8, 2.0)

ax_b.fill_between([SIG, xmax], SIG, ymax, color='#FADBD8', alpha=0.25, zorder=0)
ax_b.axvline(SIG, color='#2980B9', lw=1.2, ls='--', alpha=0.65)
ax_b.axhline(SIG, color='#E74C3C', lw=1.2, ls='--', alpha=0.65)
ax_b.text(SIG+0.04, 0.05, 'TCGA p=0.05',
          color='#2980B9', fontsize=11, rotation=90, va='bottom')
ax_b.text(0.05, SIG+0.04, 'CPTAC-3 p=0.05',
          color='#E74C3C', fontsize=11, va='bottom')

for i, row in SCREEN.iterrows():
    is_sel = bool(selected_pc and row['PC'] == selected_pc)
    t_s = row['t_sig']; e_s = row['e_sig']; bil = row['bilateral']
    if is_sel:            col = '#C0392B'
    elif bil:             col = '#27AE60'
    elif t_s and not e_s: col = '#2980B9'
    elif e_s and not t_s: col = '#9B59B6'
    else:                 col = '#BDC3C7'

    ax_b.scatter(t_lp[i], e_lp[i], s=sz[i], c=col,
                 zorder=4 if is_sel else 3,
                 edgecolors='#111' if is_sel else col,
                 linewidths=1.8 if is_sel else 0.4, alpha=0.90)
    ax_b.annotate(row['PC'],
                  xy=(t_lp[i], e_lp[i]),
                  xytext=(t_lp[i]+0.06, e_lp[i]+0.06),
                  fontsize=12.5 if is_sel else 11.5,
                  fontweight='bold' if is_sel else 'normal',
                  color='#C0392B' if is_sel else '#444')

if selected_pc and sel_row is not None:
    ax_b.annotate(
        (f'★  {selected_pc}\n'
         f'TCGA p={sel_row["t_p"]:.3f} {star(sel_row["t_p"])}\n'
         f'CPTAC-3 p={sel_row["e_p"]:.3f} {star(sel_row["e_p"])}\n'
         f'→ Bilateral + {direction_text}'),
        xy=(t_lp[selected_idx], e_lp[selected_idx]),
        xytext=(t_lp[selected_idx]+0.40, e_lp[selected_idx]+0.55),
        fontsize=11.5, color='#C0392B', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.30', facecolor='#FADBD8',
                  edgecolor='#C0392B', alpha=0.92),
        arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.8))

ax_b.set_xlabel('−log₁₀(p)  TCGA', fontsize=14)
ax_b.set_ylabel('−log₁₀(p)  CPTAC-3', fontsize=14)
ax_b.tick_params(axis='both', labelsize=12)
ax_b.set_xlim(0, xmax); ax_b.set_ylim(0, ymax)
ax_b.set_title('Bilateral Significance\nTCGA vs CPTAC-3', fontsize=14, fontweight='bold')

for sp in ['top', 'right']: ax_b.spines[sp].set_visible(False)
ax_b.text(-0.12, 1.05, 'B', transform=ax_b.transAxes, fontsize=17, fontweight='bold')


# ============================================================
# Figure title
# ============================================================
if selected_pc:
    fig_title = (
        f'Selection of the Key Prognostic Latent Feature \n'
    )
else:
    fig_title = (
        'Selection of PC6 as the Key Prognostic Latent Feature\n'
        'No component satisfied bilateral prognostic replication with concordant effect direction.'
    )

fig.suptitle(fig_title, fontsize=16, fontweight='bold', y=0.97)

plt.savefig(OUTPATH, dpi=300, bbox_inches='tight', facecolor='white', format='tiff')
plt.close()

print(f"Saved: {OUTPATH}")

Loading data...
  TCGA: 495  |  CPTAC-3: 179
Computing Cox...
  PC1: TCGA HR=1.045 p=0.5557 | CPTAC-3 HR=1.083 p=0.5862
  PC2: TCGA HR=0.866 p=0.0863 | CPTAC-3 HR=1.086 p=0.5327
  PC3: TCGA HR=1.069 p=0.4017 | CPTAC-3 HR=0.918 p=0.6157
  PC4: TCGA HR=1.124 p=0.1470 | CPTAC-3 HR=1.117 p=0.4162
  PC5: TCGA HR=0.803 p=0.0009 | CPTAC-3 HR=0.887 p=0.2759
  PC6: TCGA HR=1.191 p=0.0319 | CPTAC-3 HR=1.404 p=0.0118
  PC7: TCGA HR=0.872 p=0.0859 | CPTAC-3 HR=0.856 p=0.3368
  PC8: TCGA HR=0.813 p=0.0095 | CPTAC-3 HR=0.920 p=0.4947
  PC9: TCGA HR=1.075 p=0.3962 | CPTAC-3 HR=0.964 p=0.8167
  PC10: TCGA HR=0.896 p=0.1393 | CPTAC-3 HR=0.977 p=0.8751

Selected PC: PC6
Saved: C:\Users\nsw\Desktop\body_composition\Figure3.tiff


In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.duration.survfunc import SurvfuncRight
from scipy.stats import chi2, norm

# ============================================================
# CONFIG
# ============================================================
BASE = r"C:\Users\nsw\Desktop\body_composition"

TCGA_PATH = os.path.join(BASE, "TCGA_with_genes.csv")
EXT_PATH  = os.path.join(BASE, "External_with_genes.csv")
AE_TCGA   = os.path.join(BASE, "ae_latent_pca10_all.csv")
AE_EXT    = os.path.join(BASE, "ae_latent_pca10_external.csv")

OUTDIR = os.path.join(BASE, "clinical_impact_pc6")
os.makedirs(OUTDIR, exist_ok=True)

TAU_MONTHS = 60.0
SELECTED_PC = "PC6"


# ============================================================
# HELPERS
# ============================================================
def format_stage_binary(x):
    s = str(x).upper()
    if ("III" in s) or ("IV" in s):
        return 1.0
    if ("II" in s) or ("I" in s):
        return 0.0
    return np.nan


def preprocess_and_merge(clin_df, ae_df, cohort_name):
    df = clin_df.copy()
    ae = ae_df.copy()

    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["OS"] = pd.to_numeric(df["OS"], errors="coerce")
    df["OS_time_months"] = pd.to_numeric(df["OS_time"], errors="coerce") / 30.44
    df["SMA_zscore"] = pd.to_numeric(df["SMA_zscore"], errors="coerce")
    df["sex_bin"] = (df["sex"].astype(str).str.upper() == "MALE").astype(float)
    df["stage_bin"] = df["stage"].apply(format_stage_binary)

    if cohort_name == "TCGA":
        df["pid"] = df["patient_id"].astype(str).str[:12]
        ae["pid"] = ae["patient_id"].astype(str).str[:12]
    else:
        df["pid"] = df["patient_id"].astype(str)
        ae["pid"] = ae["patient_id"].astype(str)

    use_cols = ["pid"] + [f"AE_PC{i}" for i in range(1, 11)]
    merged = df.merge(ae[use_cols], on="pid", how="inner")

    for i in range(1, 11):
        col = f"AE_PC{i}"
        zcol = f"{col}_z"
        std = merged[col].std()
        if pd.isna(std) or std == 0:
            merged[zcol] = np.nan
        else:
            merged[zcol] = (merged[col] - merged[col].mean()) / std

    return merged


def fit_joint_cox(df, interaction=False):
    pc_col = f"AE_{SELECTED_PC}_z"
    use_cols = [
        "OS_time_months", "OS", "cancer_type",
        "age", "sex_bin", "stage_bin", "SMA_zscore", pc_col
    ]
    d = df[use_cols].dropna().copy()
    d = d[d["OS_time_months"] > 0].copy()

    if interaction:
        d["SMA_PC6_interaction"] = d["SMA_zscore"] * d[pc_col]
        exog_cols = ["SMA_zscore", pc_col, "SMA_PC6_interaction", "age", "sex_bin", "stage_bin"]
    else:
        exog_cols = ["SMA_zscore", pc_col, "age", "sex_bin", "stage_bin"]

    model = PHReg(
        endog=d["OS_time_months"],
        exog=d[exog_cols],
        status=d["OS"],
        strata=d["cancer_type"],
        ties="breslow"
    )
    result = model.fit()
    return d, result, exog_cols


def phreg_summary_table(result, exog_cols, cohort_name, model_name):
    rows = []
    for i, var in enumerate(exog_cols):
        beta = float(result.params[i])
        se = float(result.bse[i])
        z = beta / se
        p = 2 * norm.sf(abs(z))
        lo = beta - 1.96 * se
        hi = beta + 1.96 * se

        rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "variable": var,
            "beta": beta,
            "HR": np.exp(beta),
            "CI_low": np.exp(lo),
            "CI_high": np.exp(hi),
            "p_value": p
        })

    return pd.DataFrame(rows)


def lr_test(base_result, full_result, df_diff=1):
    LR = 2 * (full_result.llf - base_result.llf)
    p = chi2.sf(LR, df_diff)
    return LR, p


def km_rmst(time, event, tau):
    sf = SurvfuncRight(time, event)
    surv_times = np.asarray(sf.surv_times, dtype=float)
    surv_prob = np.asarray(sf.surv_prob, dtype=float)

    times = np.concatenate(([0.0], surv_times[surv_times < tau], [tau]))
    surv_times_lt = surv_times[surv_times < tau]
    surv_prob_lt = surv_prob[surv_times < tau]

    area = 0.0
    current_surv = 1.0

    for k in range(len(times) - 1):
        t0, t1 = times[k], times[k + 1]
        area += (t1 - t0) * current_surv
        if k < len(surv_times_lt):
            current_surv = surv_prob_lt[k]

    return area


def make_joint_groups(df, sma_cut, pc_cut):
    d = df.copy()
    pc_col = f"AE_{SELECTED_PC}_z"

    d["SMA_group"] = np.where(d["SMA_zscore"] >= sma_cut, "High SMA", "Low SMA")
    d["PC6_group"] = np.where(d[pc_col] >= pc_cut, "High PC6", "Low PC6")
    d["joint_group"] = d["SMA_group"] + " / " + d["PC6_group"]
    return d


def rmst_table(df, cohort_name, tau):
    order = [
        "High SMA / Low PC6",
        "High SMA / High PC6",
        "Low SMA / Low PC6",
        "Low SMA / High PC6",
    ]

    rows = []
    for g in order:
        dd = df[df["joint_group"] == g].copy()
        if len(dd) == 0:
            continue

        rows.append({
            "cohort": cohort_name,
            "group": g,
            "n": len(dd),
            "events": int(dd["OS"].sum()),
            f"RMST_{int(tau)}m": km_rmst(dd["OS_time_months"], dd["OS"], tau)
        })

    return pd.DataFrame(rows)


def build_risk_surface(result, exog_cols, cohort_name, df_used):
    beta_map = {var: float(result.params[i]) for i, var in enumerate(exog_cols)}
    beta_sma = beta_map["SMA_zscore"]
    beta_pc6 = beta_map[f"AE_{SELECTED_PC}_z"]
    beta_int = beta_map.get("SMA_PC6_interaction", 0.0)

    x_min = np.nanquantile(df_used["SMA_zscore"], 0.01)
    x_max = np.nanquantile(df_used["SMA_zscore"], 0.99)
    y_min = np.nanquantile(df_used[f"AE_{SELECTED_PC}_z"], 0.01)
    y_max = np.nanquantile(df_used[f"AE_{SELECTED_PC}_z"], 0.99)

    x = np.linspace(x_min, x_max, 220)
    y = np.linspace(y_min, y_max, 220)
    X, Y = np.meshgrid(x, y)

    RR = np.exp(beta_sma * X + beta_pc6 * Y + beta_int * X * Y)

    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    cf = ax.contourf(X, Y, RR, levels=14, cmap="RdYlBu_r", alpha=0.95)
    cs = ax.contour(X, Y, RR, levels=8, colors="white", linewidths=0.7, alpha=0.65)
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.2f")

    ax.scatter(
        df_used["SMA_zscore"],
        df_used[f"AE_{SELECTED_PC}_z"],
        s=16, facecolors="white", edgecolors="#555555",
        linewidths=0.35, alpha=0.35, zorder=3
    )

    ax.axvline(0, color="#333333", lw=1.1, ls="--", alpha=0.8)
    ax.axhline(0, color="#333333", lw=1.1, ls="--", alpha=0.8)
    ax.set_xlabel("SMA z-score")
    ax.set_ylabel(f"{SELECTED_PC} z-score")
    ax.set_title(f"{cohort_name}: risk surface from SMA × {SELECTED_PC} interaction model", fontweight="bold")

    txt = (
        f"Relative hazard = exp(β_SMA*SMA + β_{SELECTED_PC}*{SELECTED_PC} + β_int*SMA*{SELECTED_PC})\n"
        f"β_SMA={beta_sma:.3f}, β_{SELECTED_PC}={beta_pc6:.3f}, β_int={beta_int:.3f}"
    )
    ax.text(
        0.03, 0.97, txt,
        transform=ax.transAxes,
        ha="left", va="top", fontsize=8.8,
        bbox=dict(boxstyle="round,pad=0.30", facecolor="white", edgecolor="#BBBBBB", alpha=0.92)
    )

    cbar = plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Relative hazard")

    out_png = os.path.join(OUTDIR, f"risk_surface_{cohort_name.lower()}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

    return out_png


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    tcga = pd.read_csv(TCGA_PATH)
    ext = pd.read_csv(EXT_PATH)
    ae_t = pd.read_csv(AE_TCGA)
    ae_e = pd.read_csv(AE_EXT)

    df_t = preprocess_and_merge(tcga, ae_t, "TCGA")
    df_e = preprocess_and_merge(ext, ae_e, "CPTAC")

    # 1) Joint multivariable Cox
    d_t_joint, res_t_joint, cols_t_joint = fit_joint_cox(df_t, interaction=False)
    d_e_joint, res_e_joint, cols_e_joint = fit_joint_cox(df_e, interaction=False)

    tab_t_joint = phreg_summary_table(res_t_joint, cols_t_joint, "TCGA", "joint")
    tab_e_joint = phreg_summary_table(res_e_joint, cols_e_joint, "CPTAC", "joint")

    # 2) Interaction analysis
    d_t_int, res_t_int, cols_t_int = fit_joint_cox(df_t, interaction=True)
    d_e_int, res_e_int, cols_e_int = fit_joint_cox(df_e, interaction=True)

    tab_t_int = phreg_summary_table(res_t_int, cols_t_int, "TCGA", "interaction")
    tab_e_int = phreg_summary_table(res_e_int, cols_e_int, "CPTAC", "interaction")

    lr_t, lr_p_t = lr_test(res_t_joint, res_t_int, df_diff=1)
    lr_e, lr_p_e = lr_test(res_e_joint, res_e_int, df_diff=1)

    interaction_lr = pd.DataFrame([
        {"cohort": "TCGA",  "LR_stat": lr_t, "LR_p_value": lr_p_t},
        {"cohort": "CPTAC", "LR_stat": lr_e, "LR_p_value": lr_p_e},
    ])

    # 3) RMST (use TCGA medians as common cutoffs)
    sma_cut = float(d_t_joint["SMA_zscore"].median())
    pc_cut = float(d_t_joint[f"AE_{SELECTED_PC}_z"].median())

    rmst_t = rmst_table(make_joint_groups(d_t_joint, sma_cut, pc_cut), "TCGA", TAU_MONTHS)
    rmst_e = rmst_table(make_joint_groups(d_e_joint, sma_cut, pc_cut), "CPTAC", TAU_MONTHS)
    rmst_all = pd.concat([rmst_t, rmst_e], ignore_index=True)

    # 4) Risk surface modeling (interaction model)
    risk_png_t = build_risk_surface(res_t_int, cols_t_int, "TCGA", d_t_int)
    risk_png_e = build_risk_surface(res_e_int, cols_e_int, "CPTAC", d_e_int)

    # save tables
    cox_all = pd.concat([tab_t_joint, tab_e_joint, tab_t_int, tab_e_int], ignore_index=True)
    cox_all.to_csv(os.path.join(OUTDIR, "cox_summary_pc6.csv"), index=False)
    interaction_lr.to_csv(os.path.join(OUTDIR, "interaction_lr_test_pc6.csv"), index=False)
    rmst_all.to_csv(os.path.join(OUTDIR, f"rmst_{int(TAU_MONTHS)}m_pc6.csv"), index=False)

    print("Saved:")
    print(os.path.join(OUTDIR, "cox_summary_pc6.csv"))
    print(os.path.join(OUTDIR, "interaction_lr_test_pc6.csv"))
    print(os.path.join(OUTDIR, f"rmst_{int(TAU_MONTHS)}m_pc6.csv"))
    print(risk_png_t)
    print(risk_png_e)

Saved:
C:\Users\nsw\Desktop\body_composition\clinical_impact_pc6\cox_summary_pc6.csv
C:\Users\nsw\Desktop\body_composition\clinical_impact_pc6\interaction_lr_test_pc6.csv
C:\Users\nsw\Desktop\body_composition\clinical_impact_pc6\rmst_60m_pc6.csv
C:\Users\nsw\Desktop\body_composition\clinical_impact_pc6\risk_surface_tcga.png
C:\Users\nsw\Desktop\body_composition\clinical_impact_pc6\risk_surface_cptac.png


In [24]:
"""
Figure 4. Clinical Impact of PC6 in Combination with Body Composition
======================================================================
Panel A: TCGA multivariable Cox (SMA + PC6 forest plot)
Panel B: CPTAC-3 multivariable Cox (동일 구조)
Panel C: Risk surface (SMA × PC6, TCGA)
         - y축: -3~3 고정
         - x축: 데이터 범위 (그림 중앙 배치)
         - Highest risk 박스: 왼쪽 상단
         - Relative hazard 박스: 오른쪽 상단
         - Lowest risk 박스: 우하단

입력:
  TCGA_with_genes.csv
  External_with_genes.csv
  ae_latent_pca10_all.csv
  ae_latent_pca10_external.csv

출력:
  Figure4.tiff  (dpi=300)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR  = r"C:\Users\nsw\Desktop\body_composition"
TCGA_FILE = os.path.join(BASE_DIR, "TCGA_with_genes.csv")
CPTAC_FILE= os.path.join(BASE_DIR, "External_with_genes.csv")
AE_TCGA   = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
AE_CPTAC  = os.path.join(BASE_DIR, "ae_latent_pca10_external.csv")
OUT_TIFF  = os.path.join(BASE_DIR, "Figure4.tiff")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드 & 전처리                        ║
# ╚══════════════════════════════════════════════════════════════╝

tcga     = pd.read_csv(TCGA_FILE)
cptac    = pd.read_csv(CPTAC_FILE)
pc_tcga  = pd.read_csv(AE_TCGA)
pc_cptac = pd.read_csv(AE_CPTAC)

def clean_stage(s):
    if pd.isna(s): return np.nan
    s = str(s).upper()
    if 'IV'  in s: return 4
    if 'III' in s: return 3
    if 'II'  in s: return 2
    if 'I'   in s: return 1
    return np.nan

for df in [tcga, cptac]:
    df['stage_num'] = df['stage'].apply(clean_stage)
    df['sex_num']   = (df['sex'].astype(str).str.upper() == 'MALE').astype(float)

tcga['pid']      = tcga['patient_id'].astype(str).str[:12]
pc_tcga['pid']   = pc_tcga['patient_id'].astype(str).str[:12]
cptac['pid']     = cptac['patient_id'].astype(str)
pc_cptac['pid']  = pc_cptac['patient_id'].astype(str)

m_tcga  = pc_tcga.merge(
    tcga[['pid','OS_time','OS','cancer_type','age','stage_num','sex_num','SMA_zscore']],
    on='pid')
m_cptac = pc_cptac.merge(
    cptac[['pid','OS_time','OS','cancer_type','age','stage_num','sex_num','SMA_zscore']],
    on='pid')

for df in [m_tcga, m_cptac]:
    df['PC6_z'] = (df['AE_PC6'] - df['AE_PC6'].mean()) / df['AE_PC6'].std()

def pstar(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ╔══════════════════════════════════════════════════════════════╗
# ║          Cox: SMA + PC6, adj age/sex/stage, strat cancer     ║
# ╚══════════════════════════════════════════════════════════════╝

def run_cox(df):
    covs = ['SMA_zscore', 'PC6_z', 'age', 'stage_num']
    if df['sex_num'].std() > 0.05:
        covs.append('sex_num')
    d = df[['OS_time','OS','cancer_type'] + covs].dropna()
    d = d[d['OS_time'] > 0].copy()
    cph = CoxPHFitter()
    cph.fit(d, duration_col='OS_time', event_col='OS',
            strata='cancer_type', robust=True)
    return cph.summary, d, cph

print("Fitting Cox models...")
res_t, d_t, cph_t = run_cox(m_tcga)
res_c, d_c, cph_c = run_cox(m_cptac)

print(f"TCGA  n={len(d_t)}, events={int(d_t['OS'].sum())}")
print(f"CPTAC n={len(d_c)}, events={int(d_c['OS'].sum())}")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃                             ║
# ╚══════════════════════════════════════════════════════════════╝

LS, TS, TIS, PS = 11, 9.5, 12, 14
VAR_COLORS = {'SMA_zscore': '#2166AC', 'PC6_z': '#D73027'}
VAR_LABELS = {'SMA_zscore': 'SMA z-score', 'PC6_z': 'PC6 (z-score)'}
VAR_MK     = {'SMA_zscore': 'D', 'PC6_z': 's'}

fig = plt.figure(figsize=(20, 7))
fig.patch.set_facecolor('white')
gs = GridSpec(1, 3, figure=fig,
              wspace=0.38, left=0.05, right=0.97,
              top=0.90, bottom=0.10,
              width_ratios=[1, 1, 1.2])

# ════════════════════════════════════════════════════════════════
# Forest plot helper (A, B 공통)
# ════════════════════════════════════════════════════════════════

def draw_forest(ax, summary, cohort, panel_label, n, events):
    X_MIN = 0.25; X_MAX = 2.30
    X_CI_START = 0.55    # CI 시작 x (라벨 영역과 분리)
    LABEL_X = 0.28       # 라벨/텍스트 x 위치

    y_pos = {'SMA_zscore': 2.0, 'PC6_z': 0.0}

    ax.axvline(1.0, color='#888', lw=1.0, ls='--', alpha=0.6, zorder=1)
    ax.axhline(1.0, color='#e0e0e0', lw=0.9, zorder=0)

    for var in ['SMA_zscore', 'PC6_z']:
        r      = summary.loc[var]
        hr     = r['exp(coef)']
        lo     = r['exp(coef) lower 95%']
        hi     = r['exp(coef) upper 95%']
        p      = r['p']
        y      = y_pos[var]
        col    = VAR_COLORS[var]
        is_sig = (p < 0.05)

        # 배경 shading
        ax.axhspan(y - 0.9, y + 0.9, color=col, alpha=0.05, zorder=0)

        # variable 라벨 — 왼쪽 고정
        ax.text(LABEL_X, y + 0.35, VAR_LABELS[var],
                fontsize=11, va='center', ha='left',
                fontweight='bold', color=col)

        # CI line (라벨 영역 이후부터)
        lo_plot = max(lo, X_CI_START)
        ax.plot([lo_plot, hi], [y, y], color=col, lw=2.8, alpha=0.9, zorder=2)

        # point (SMA=다이아몬드, PC6=사각형)
        mk = VAR_MK[var]
        ax.scatter(hr, y, color=col, s=120, marker=mk, zorder=3,
                   edgecolors='white', linewidths=0.8)

        # HR 텍스트 — 라벨 아래 (겹침 방지)
        ax.text(LABEL_X, y - 0.20,
                f"HR={hr:.3f} ({lo:.3f}–{hi:.3f})",
                fontsize=8.5, va='top', ha='left',
                color=col if is_sig else '#999',
                fontweight='bold' if is_sig else 'normal')
        ax.text(LABEL_X, y - 0.62,
                f"p={p:.4f} {pstar(p)}",
                fontsize=8.5, va='top', ha='left',
                color=col if is_sig else '#999',
                fontweight='bold' if is_sig else 'normal')

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(-0.9, 3.0)
    ax.set_yticks([])
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=LS)
    ax.set_title(f'({panel_label}) {cohort}: multivariable Cox model\n'
                 f'n={n}, events={events} | adj: age, sex, stage | strat: cancer type',
                 fontsize=10.5, fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(axis='x', labelsize=TS)
    ax.text(-0.10, 1.06, panel_label, transform=ax.transAxes,
            fontsize=PS, fontweight='bold')

    h1 = Line2D([0],[0], marker='D', color='w', markerfacecolor='#2166AC',
                markeredgecolor='white', markersize=9, label='SMA z-score')
    h2 = Line2D([0],[0], marker='s', color='w', markerfacecolor='#D73027',
                markeredgecolor='white', markersize=9, label='PC6')
    ax.legend(handles=[h1, h2], fontsize=9, frameon=True,
              loc='lower right').get_frame().set_edgecolor('#ccc')

# ── A: TCGA ───────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
draw_forest(ax_a, res_t, 'TCGA', 'A', len(d_t), int(d_t['OS'].sum()))

# ── B: CPTAC-3 ────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
draw_forest(ax_b, res_c, 'CPTAC-3', 'B', len(d_c), int(d_c['OS'].sum()))

# ════════════════════════════════════════════════════════════════
# Panel C: Risk surface
# ════════════════════════════════════════════════════════════════

ax_c = fig.add_subplot(gs[0, 2])

covs_g = ['SMA_zscore', 'PC6_z', 'age', 'stage_num']
if m_tcga['sex_num'].std() > 0.05:
    covs_g.append('sex_num')

# x: 데이터 범위, y: -3~3 고정
sma_q01 = m_tcga['SMA_zscore'].quantile(0.01)
sma_q99 = m_tcga['SMA_zscore'].quantile(0.99)

sma_grid = np.linspace(sma_q01, sma_q99, 80)
pc6_grid = np.linspace(-3, 3, 80)
SMA_G, PC6_G = np.meshgrid(sma_grid, pc6_grid)

grid_df = pd.DataFrame({
    'SMA_zscore': SMA_G.ravel(),
    'PC6_z'     : PC6_G.ravel(),
    'age'       : d_t['age'].median(),
    'stage_num' : d_t['stage_num'].median(),
})
if 'sex_num' in covs_g:
    grid_df['sex_num'] = d_t['sex_num'].median()

params = cph_t.params_
lp = grid_df[covs_g].values @ params[covs_g].values
RH = np.exp(lp.reshape(SMA_G.shape))

# filled contour
im = ax_c.contourf(SMA_G, PC6_G, RH, levels=30, cmap='RdYlBu_r', alpha=0.92)

# white contour lines + labels
cs = ax_c.contour(SMA_G, PC6_G, RH, levels=8,
                   colors='white', linewidths=0.8, alpha=0.7)
ax_c.clabel(cs, inline=True, fontsize=7.5, fmt='%.2f', colors='white')

# colorbar
cb = fig.colorbar(im, ax=ax_c, shrink=0.85, pad=0.02)
cb.set_label('Relative hazard', fontsize=9)
cb.ax.tick_params(labelsize=8)

# patient scatter
ax_c.scatter(m_tcga['SMA_zscore'], m_tcga['PC6_z'],
             c='white', s=8, alpha=0.35, linewidths=0, zorder=3)

# reference lines
ax_c.axvline(0, color='#333', lw=1.2, ls='--', alpha=0.8)
ax_c.axhline(0, color='#333', lw=1.2, ls='--', alpha=0.8)

# 축 범위 설정
ax_c.set_xlim(sma_q01 - 0.3, sma_q99 + 0.3)
ax_c.set_ylim(-3, 3)

# ── 텍스트 박스 배치 (data 좌표계) ────────────────────────
# 1) Highest risk — 왼쪽 상단 (그림 안, 겹침 없음)
ax_c.text(sma_q01, 2.75,
          'Highest risk\nLow SMA\nHigh PC6',
          ha='left', va='top',
          fontsize=8, color='white', fontweight='bold',
          bbox=dict(boxstyle='round,pad=0.3', fc='#8B0000', alpha=0.88),
          zorder=5)

# 2) Relative hazard — 오른쪽으로 밀어서 (SMA=0.3)
ax_c.text(0.3, 2.75,
          'Relative hazard (TCGA Cox model)\nRR = exp(β_SMA·SMA_z + β_PC6·PC6_z)',
          ha='left', va='top',
          fontsize=7.5, color='#222',
          bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#bbb', alpha=0.92),
          zorder=5)

# 3) Lowest risk — 우하단
ax_c.text(0.98, 0.03,
          'Lowest risk\nHigh SMA\nLow PC6',
          transform=ax_c.transAxes, ha='right', va='bottom',
          fontsize=8, color='white', fontweight='bold',
          bbox=dict(boxstyle='round,pad=0.25', fc='#08519C', alpha=0.85),
          zorder=5)

ax_c.set_xlabel('SMA z-score', fontsize=LS)
ax_c.set_ylabel('PC6 z-score', fontsize=LS)
ax_c.set_title('Risk Surface: SMA × PC6\n(TCGA-centered joint model)',
               fontsize=TIS, fontweight='bold')
ax_c.tick_params(labelsize=TS)
ax_c.spines[['top','right']].set_visible(False)
ax_c.text(-0.12, 1.06, 'C', transform=ax_c.transAxes,
          fontsize=PS, fontweight='bold')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle(
    'Clinical Impact of PC6 in Combination with Body Composition',
    fontsize=11.5, fontweight='bold', y=1.01)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Fitting Cox models...
TCGA  n=491, events=190
CPTAC n=178, events=72

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure4.tiff


In [25]:
"""
Figure 5. PC6 Represents a Conserved Warburg-like Metabolic Program
=====================================================================
Panel A: TCGA — gene-level Spearman correlation with PC6
Panel B: CPTAC-3 — same analysis
Panel C: Pathway-level mean r (TCGA vs CPTAC-3)
Panel D: Warburg vs Proliferation vs OXPHOS comparison (TCGA)
Panel E: ssGSEA NES ~ PC6 Spearman (TCGA vs CPTAC-3)

입력:
  RNA_Group1~4_updated_to_new_symbol.csv  (TCGA)
  ex_RNA_Group1~4.csv                     (CPTAC-3)
  ae_latent_pca10_all.csv
  ae_latent_pca10_external.csv

출력:
  Figure5.tiff

패키지:
  pip install gseapy
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR = r"C:\Users\nsw\Desktop\body_composition"

TCGA_RNA_FILES = [
    os.path.join(BASE_DIR, "RNA_Group1_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group2_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group3_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group4_updated_to_new_symbol.csv"),
]
CPTAC_RNA_FILES = [
    os.path.join(BASE_DIR, "ex_RNA_Group1_LowMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group2_LowMuscle_HighImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group3_HighMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group4_HighMuscle_HighImmune.csv"),
]
TCGA_PC_FILE  = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
CPTAC_PC_FILE = os.path.join(BASE_DIR, "ae_latent_pca10_external.csv")
OUT_TIFF      = os.path.join(BASE_DIR, "Figure5.tiff")
OUT_SSGSEA    = os.path.join(BASE_DIR, "results_ssgsea_pc6.csv")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    유전자 세트 정의                             ║
# ╚══════════════════════════════════════════════════════════════╝

GENE_SETS = {
    'Glycolysis'   : ['HK2','PFKP','ALDOA','ENO1','PGK1','PKM','LDHA','GAPDH','TPI1','PGAM1'],
    'Hypoxia'      : ['VEGFA','NDRG1','CA9','HIF1A','ADM','HMOX1'],
    'Lactate/MCT'  : ['LDHA','LDHB','SLC16A3','SLC16A1'],
    'OXPHOS'       : ['SDHA','SDHB','COX5A','NDUFB3','UQCRC1'],
    'Proliferation': ['E2F1','CDK4','CCND1','CDK1','CCNA2','TOP2A','MKI67'],
}

SET_COLORS = {
    'Glycolysis'   : '#E74C3C',
    'Hypoxia'      : '#E67E22',
    'Lactate/MCT'  : '#C0392B',
    'OXPHOS'       : '#2980B9',
    'Proliferation': '#7F8C8D',
}

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드 & 전처리                        ║
# ╚══════════════════════════════════════════════════════════════╝

def merge_rna_groups(files):
    dfs = [pd.read_csv(f).set_index('gene') for f in files]
    merged = pd.concat(dfs, axis=1)
    return merged.loc[~merged.index.duplicated(keep='first')]

print("Loading RNA data...")
rna_tcga  = merge_rna_groups(TCGA_RNA_FILES)
rna_cptac = merge_rna_groups(CPTAC_RNA_FILES)
rna_tcga.columns = [c[:12] for c in rna_tcga.columns]

pc_tcga  = pd.read_csv(TCGA_PC_FILE)
pc_cptac = pd.read_csv(CPTAC_PC_FILE)
pc_tcga['pid']  = pc_tcga['patient_id'].astype(str).str[:12]
pc_cptac['pid'] = pc_cptac['patient_id'].astype(str)

for df in [pc_tcga, pc_cptac]:
    df['PC6_z'] = (df['AE_PC6'] - df['AE_PC6'].mean()) / df['AE_PC6'].std()

def align_pc6_rna(pc_df, rna_df):
    common  = list(set(pc_df['pid']) & set(rna_df.columns))
    pc_sub  = pc_df.set_index('pid').loc[common, 'PC6_z']
    rna_sub = rna_df[common].T.astype(float)
    rna_sub = np.log2(rna_sub + 1)
    return pc_sub, rna_sub

pc6_t, rna_t = align_pc6_rna(pc_tcga,  rna_tcga)
pc6_c, rna_c = align_pc6_rna(pc_cptac, rna_cptac)

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Gene-level Spearman                        ║
# ╚══════════════════════════════════════════════════════════════╝

def compute_gene_corr(pc6, rna):
    all_genes = list({g for genes in GENE_SETS.values() for g in genes})
    results = {}
    for gene in all_genes:
        if gene not in rna.columns: continue
        x = pc6.values.astype(float)
        y = rna[gene].values.astype(float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 10: continue
        r, p = spearmanr(x[mask], y[mask])
        results[gene] = {'r': r, 'p': p}
    df = pd.DataFrame(results).T
    _, q, _, _ = multipletests(df['p'].values, method='fdr_bh')
    df['q'] = q
    return df

print("Computing gene-level Spearman correlations...")
corr_t = compute_gene_corr(pc6_t, rna_t)
corr_c = compute_gene_corr(pc6_c, rna_c)

def pathway_summary(corr_df):
    rows = []
    for name, genes in GENE_SETS.items():
        avail = [g for g in genes if g in corr_df.index]
        if not avail: continue
        rows.append({'pathway': name, 'mean_r': corr_df.loc[avail,'r'].mean(), 'n': len(avail)})
    return pd.DataFrame(rows)

summ_t = pathway_summary(corr_t)
summ_c = pathway_summary(corr_c)

# ╔══════════════════════════════════════════════════════════════╗
# ║          ssGSEA enrichment score ~ PC6  (Panel E)            ║
# ╚══════════════════════════════════════════════════════════════╝

print("\nRunning ssGSEA...")
ssgsea_rows   = []
ssgsea_results = {}
RUN_SSGSEA    = False

try:
    import gseapy as gp

    gene_set_upper = {k: [g.upper() for g in v] for k, v in GENE_SETS.items()}

    for coh_name, rna_df, pc6 in [('TCGA', rna_t, pc6_t),
                                    ('CPTAC-3', rna_c, pc6_c)]:
        expr = rna_df.T.copy()
        expr.index = expr.index.str.upper()

        ss = gp.ssgsea(
            data=expr,
            gene_sets=gene_set_upper,
            outdir=None,
            sample_norm_method='rank',
            no_plot=True,
            min_size=2,
            verbose=False
        )

        scores = ss.res2d.pivot(index='Term', columns='Name', values='NES').T.astype(float)
        common  = list(set(scores.index) & set(pc6.index))
        sc_sub  = scores.loc[common]
        pc6_sub = pc6.loc[common].astype(float)

        coh_pvals = []; coh_data = []
        for pathway in GENE_SETS.keys():
            if pathway not in sc_sub.columns: continue
            x = pc6_sub.values.astype(float)
            y = sc_sub[pathway].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() < 5: continue
            r, p = spearmanr(x[mask], y[mask])
            coh_pvals.append(p)
            coh_data.append({'cohort': coh_name, 'pathway': pathway,
                              'r': round(r, 3), 'p': round(p, 4), 'n': int(mask.sum())})

        if coh_pvals:
            _, qvals, _, _ = multipletests(coh_pvals, method='fdr_bh')
            for i, row in enumerate(coh_data):
                row['q'] = round(qvals[i], 4)
                row['sig'] = ('***' if row['q']<0.001 else '**' if row['q']<0.01
                               else '*' if row['q']<0.05 else 'ns')
                ssgsea_rows.append(row)

        # ★ 핵심: pathway를 index로 설정해서 저장
        ssgsea_results[coh_name] = (pd.DataFrame(coh_data).set_index('pathway')
                                     if coh_data else pd.DataFrame())

    pd.DataFrame(ssgsea_rows).to_csv(OUT_SSGSEA, index=False)
    RUN_SSGSEA = True
    print("  ssGSEA complete!")

except ImportError:
    print("  gseapy 미설치. pip install gseapy 후 재실행.")
except Exception as e:
    print(f"  ssGSEA 오류: {e}")

# ╔══════════════════════════════════════════════════════════════╗
# ║   Figure 레이아웃: 3행 × 2열                                   ║
# ║   Row 0: A, B  (gene-level barplot)                           ║
# ║   Row 1: C, D  (pathway mean r / dot plot)                    ║
# ║   Row 2: E     (ssGSEA, full width)                           ║
# ╚══════════════════════════════════════════════════════════════╝

fig = plt.figure(figsize=(22, 15))
fig.patch.set_facecolor('white')

gs = GridSpec(3, 2, figure=fig,
              hspace=0.48, wspace=0.35,
              left=0.07, right=0.97, top=0.93, bottom=0.06,
              height_ratios=[1.2, 1.0, 0.9])

LS, TS, TIS, PS = 11, 9.0, 11.5, 14

# ── A & B ─────────────────────────────────────────────────
def draw_gene_corr(ax, corr_df, title, panel_label):
    all_genes, all_r, all_colors = [], [], []
    for name, genes in GENE_SETS.items():
        avail = [g for g in genes if g in corr_df.index]
        for g in avail:
            all_genes.append(g)
            all_r.append(corr_df.loc[g, 'r'])
            all_colors.append(SET_COLORS[name])

    y = np.arange(len(all_genes))
    ax.barh(y, all_r, color=all_colors, alpha=0.82, edgecolor='white', height=0.72)

    pvals = [corr_df.loc[g, 'p'] for g in all_genes]
    _, qvals, _, _ = multipletests(pvals, method='fdr_bh')
    for yi, q in enumerate(qvals):
        mk = '***' if q<0.001 else '**' if q<0.01 else '*' if q<0.05 else ''
        if mk:
            xpos = all_r[yi] + (0.01 if all_r[yi] >= 0 else -0.01)
            ax.text(xpos, yi, mk, fontsize=7, va='center',
                    ha='left' if all_r[yi] >= 0 else 'right',
                    color='#333', fontweight='bold')

    ax.set_yticks(y)
    ax.set_yticklabels(all_genes, fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel("Spearman r  (vs PC6)", fontsize=LS)
    ax.set_title(title, fontsize=TIS, fontweight='bold')
    ax.tick_params(axis='x', labelsize=TS)
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(-0.6, 0.7)
    handles = [mpatches.Patch(color=SET_COLORS[n], label=n) for n in GENE_SETS]
    leg = ax.legend(handles=handles, fontsize=8, frameon=True, loc='lower right')
    leg.get_frame().set_edgecolor('#ccc')
    ax.text(-0.10, 1.04, panel_label, transform=ax.transAxes,
            fontsize=PS, fontweight='bold')

ax_a = fig.add_subplot(gs[0, 0])
draw_gene_corr(ax_a, corr_t, 'TCGA: Gene-level correlation with PC6', 'A')
ax_b = fig.add_subplot(gs[0, 1])
draw_gene_corr(ax_b, corr_c, 'CPTAC-3: Gene-level correlation with PC6', 'B')

# ── C ─────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
pathways = summ_t['pathway'].tolist()
x = np.arange(len(pathways)); w = 0.35
b1 = ax_c.bar(x-w/2, summ_t['mean_r'], w,
              color=[SET_COLORS[p] for p in pathways],
              alpha=0.90, edgecolor='white', label='TCGA')
b2 = ax_c.bar(x+w/2, summ_c['mean_r'], w,
              color=[SET_COLORS[p] for p in pathways],
              alpha=0.48, edgecolor='#555', linewidth=0.7,
              hatch='//', label='CPTAC-3')
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax_c.text(bar.get_x()+bar.get_width()/2,
              h+(0.005 if h>=0 else -0.005),
              f'{h:.2f}', ha='center',
              va='bottom' if h>=0 else 'top',
              fontsize=8, fontweight='bold')
ax_c.axhline(0, color='black', lw=0.8)
ax_c.set_xticks(x)
ax_c.set_xticklabels(pathways, fontsize=9.5, rotation=15, ha='right')
ax_c.set_ylabel("Mean Spearman r  (vs PC6)", fontsize=LS)
ax_c.set_title('Pathway-level mean r (TCGA vs CPTAC-3)', fontsize=TIS, fontweight='bold')
ax_c.legend(fontsize=9.5, frameon=True)
ax_c.spines[['top','right']].set_visible(False)
ax_c.tick_params(axis='y', labelsize=TS)
ax_c.text(-0.10, 1.04, 'C', transform=ax_c.transAxes, fontsize=PS, fontweight='bold')

# ── D ─────────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
warburg_genes = (GENE_SETS['Glycolysis'] +
                 GENE_SETS['Hypoxia'] +
                 GENE_SETS['Lactate/MCT'])
np.random.seed(42)
for yi, (genes, col) in enumerate([(warburg_genes,            '#E74C3C'),
                                    (GENE_SETS['Proliferation'],'#7F8C8D'),
                                    (GENE_SETS['OXPHOS'],       '#2980B9')]):
    for gene in genes:
        if gene not in corr_t.index: continue
        r = corr_t.loc[gene, 'r']
        ax_d.scatter(r, yi+np.random.uniform(-0.15,0.15),
                     color=col, s=55, alpha=0.75,
                     edgecolors='white', linewidths=0.4, zorder=3)
        ax_d.text(r, yi+0.18, gene, fontsize=7, ha='center',
                  va='bottom', color=col, rotation=45)
    rs = [corr_t.loc[g,'r'] for g in genes if g in corr_t.index]
    if rs:
        ax_d.plot([np.mean(rs)],[yi], marker='|', color=col,
                  markersize=20, markeredgewidth=3, zorder=5)
        ax_d.text(np.mean(rs), yi-0.25,
                  f'mean r = {np.mean(rs):.2f}',
                  ha='center', va='top', fontsize=8.5,
                  color=col, fontweight='bold')
ax_d.axvline(0, color='black', lw=0.8, ls='--', alpha=0.6)
ax_d.set_yticks([0,1,2])
ax_d.set_yticklabels(['Warburg\n(Glycolysis + Hypoxia + MCT)',
                       'Proliferation\n(E2F / G2M / MKI67)',
                       'OXPHOS'], fontsize=9.5)
ax_d.set_xlabel("Spearman r  (vs PC6, TCGA)", fontsize=LS)
ax_d.set_title('PC6 ~ Warburg vs Proliferation vs OXPHOS\n'
               '(TCGA, proliferation as negative control)',
               fontsize=TIS, fontweight='bold')
ax_d.set_xlim(-0.7, 0.75); ax_d.set_ylim(-0.55, 2.7)
ax_d.spines[['top','right']].set_visible(False)
ax_d.tick_params(axis='x', labelsize=TS)
ax_d.text(-0.10, 1.04, 'D', transform=ax_d.transAxes, fontsize=PS, fontweight='bold')

# ── E  ssGSEA (full width) ────────────────────────────────
ax_e = fig.add_subplot(gs[2, :])

if RUN_SSGSEA and ssgsea_results:
    ss_t = ssgsea_results.get('TCGA', pd.DataFrame())
    ss_c = ssgsea_results.get('CPTAC-3', pd.DataFrame())

    pathways_e = list(GENE_SETS.keys())
    x_e = np.arange(len(pathways_e)); w_e = 0.32
    colors_e = [SET_COLORS[p] for p in pathways_e]

    def sig_label(q):
        return '***' if q<0.001 else '**' if q<0.01 else '*' if q<0.05 else 'ns'

    r_t_vals = [ss_t.loc[p,'r'] if p in ss_t.index else 0 for p in pathways_e]
    r_c_vals = [ss_c.loc[p,'r'] if p in ss_c.index else 0 for p in pathways_e]
    q_t_vals = [ss_t.loc[p,'q'] if p in ss_t.index else 1 for p in pathways_e]
    q_c_vals = [ss_c.loc[p,'q'] if p in ss_c.index else 1 for p in pathways_e]

    b3 = ax_e.bar(x_e-w_e/2, r_t_vals, w_e,
                  color=colors_e, alpha=0.90, edgecolor='white', label='TCGA')
    b4 = ax_e.bar(x_e+w_e/2, r_c_vals, w_e,
                  color=colors_e, alpha=0.48, edgecolor='#555',
                  linewidth=0.7, hatch='//', label='CPTAC-3')

    for i, p in enumerate(pathways_e):
        r_t=r_t_vals[i]; q_t=q_t_vals[i]
        r_c=r_c_vals[i]; q_c=q_c_vals[i]
        ct='#c0392b' if q_t<0.05 else '#888'
        cc='#c0392b' if q_c<0.05 else '#888'
        ax_e.text(x_e[i]-w_e/2, r_t+(0.012 if r_t>=0 else -0.025),
                  sig_label(q_t), ha='center',
                  va='bottom' if r_t>=0 else 'top',
                  fontsize=11, color=ct, fontweight='bold')
        ax_e.text(x_e[i]+w_e/2, r_c+(0.012 if r_c>=0 else -0.025),
                  sig_label(q_c), ha='center',
                  va='bottom' if r_c>=0 else 'top',
                  fontsize=11, color=cc, fontweight='bold')
        if abs(r_t) > 0.06:
            ax_e.text(x_e[i]-w_e/2, r_t/2, f'{r_t:+.2f}',
                      ha='center', va='center', fontsize=9,
                      color='white', fontweight='bold')
        if abs(r_c) > 0.06:
            ax_e.text(x_e[i]+w_e/2, r_c/2, f'{r_c:+.2f}',
                      ha='center', va='center', fontsize=9,
                      color='white', fontweight='bold')

    ax_e.axhline(0, color='black', lw=0.8)
    ax_e.set_xticks(x_e)
    ax_e.set_xticklabels(pathways_e, fontsize=12)
    ax_e.set_ylabel("Spearman r  (ssGSEA NES vs PC6)", fontsize=LS)
    ax_e.set_title(
        'ssGSEA Pathway Enrichment Score ~ PC6  (Panel E)\n'
        'Single-sample GSEA, rank normalization | FDR-corrected (Benjamini-Hochberg)',
        fontsize=TIS, fontweight='bold')
    ax_e.legend(fontsize=10, frameon=True,
                loc='upper right').get_frame().set_edgecolor('#ccc')
    ax_e.spines[['top','right']].set_visible(False)
    ax_e.tick_params(axis='y', labelsize=TS)
    ax_e.set_ylim(-0.25, 0.88)

else:
    ax_e.text(0.5, 0.5,
              'ssGSEA not available.\nInstall gseapy: pip install gseapy',
              ha='center', va='center', fontsize=14, color='gray',
              transform=ax_e.transAxes)
    ax_e.set_axis_off()

ax_e.text(-0.04, 1.04, 'E', transform=ax_e.transAxes,
          fontsize=PS, fontweight='bold')

# ╔══════════════════════════════════════════════════════════════╗
# ║            제목만, 부연설명 없음                                ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle(
    'PC6 Represents a Conserved Warburg-like Metabolic Program',
    fontsize=12.5, fontweight='bold', y=0.99)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Loading RNA data...


2026-04-01 14:21:19,724 [WARNING] Input data contains NA, filled NA with 0


Computing gene-level Spearman correlations...

Running ssGSEA...
  ssGSEA complete!

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure5.tiff


In [26]:
"""
Figure 6. Joint Effects of PC6 and Body Composition  (Standalone)
==================================================================
사전 계산 CSV 없이 전부 인라인으로 계산합니다.

Panel A: PC6 × SMA 4-group KM survival
Panel B: Volcano plot (PC6-High/SMA-Low vs Others)
Panel C: GSEA NES barplot (Hallmark + Immune gene sets)
Panel D: Transcriptional signature heatmap (4 groups)
Panel E: Representative marker violin (4 groups)

입력 (BASE_DIR 안에 있어야 함):
  TCGA_with_genes.csv
  ae_latent_pca10_all.csv
  RNA_Group1~4_updated_to_new_symbol.csv  (4개)

출력:
  Figure6.tiff  (dpi=300)

패키지:
  pip install lifelines scipy statsmodels
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from scipy.stats import kruskal, ranksums
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR = r"C:\Users\nsw\Desktop\body_composition"

TCGA_FILE    = os.path.join(BASE_DIR, "TCGA_with_genes.csv")
PC_TCGA_FILE = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
TCGA_RNA_FILES = [
    os.path.join(BASE_DIR, f"RNA_Group{i}_updated_to_new_symbol.csv")
    for i in range(1, 5)
]
OUT_TIFF = os.path.join(BASE_DIR, "Figure6.tiff")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드                                 ║
# ╚══════════════════════════════════════════════════════════════╝

print("Loading data...")
dfs = [pd.read_csv(f).set_index('gene') for f in TCGA_RNA_FILES]
rna = pd.concat(dfs, axis=1).loc[lambda x: ~x.index.duplicated()]
rna.columns = [c[:12] for c in rna.columns]
rna_log = np.log2(rna + 1)

pc   = pd.read_csv(PC_TCGA_FILE)
clin = pd.read_csv(TCGA_FILE)
pc['pid']   = pc['patient_id'].astype(str).str[:12]
clin['pid'] = clin['patient_id'].astype(str).str[:12]
pc['PC6_z'] = (pc['AE_PC6'] - pc['AE_PC6'].mean()) / pc['AE_PC6'].std()

merged = clin[['pid','OS_time','OS','SMA_zscore','cancer_type']].merge(
    pc[['pid','PC6_z']], on='pid')
merged['OS_months'] = merged['OS_time'] / 30.44
merged = merged.dropna(subset=['PC6_z','SMA_zscore','OS_months','OS'])

pc6_med = merged['PC6_z'].median()
sma_med = merged['SMA_zscore'].median()
merged['PC6_grp'] = np.where(merged['PC6_z'] >= pc6_med, 'PC6-High', 'PC6-Low')
merged['SMA_grp'] = np.where(merged['SMA_zscore'] >= sma_med, 'SMA-High', 'SMA-Low')
merged['group4']  = merged['PC6_grp'] + ' / ' + merged['SMA_grp']

G_ORDER  = ['PC6-Low / SMA-High','PC6-Low / SMA-Low',
            'PC6-High / SMA-High','PC6-High / SMA-Low']
G_LABELS = ['PC6-Low\nSMA-High','PC6-Low\nSMA-Low',
            'PC6-High\nSMA-High','PC6-High\nSMA-Low']
G_COLORS = {'PC6-Low / SMA-High':'#2166AC','PC6-Low / SMA-Low':'#74ADD1',
            'PC6-High / SMA-High':'#F46D43','PC6-High / SMA-Low':'#D73027'}

# ╔══════════════════════════════════════════════════════════════╗
# ║               DEG 계산 (PC6-High/SMA-Low vs Others)           ║
# ╚══════════════════════════════════════════════════════════════╝

print("Computing DEG...")
worst_pids = set(merged[merged['group4']=='PC6-High / SMA-Low']['pid'])
other_pids = set(merged[merged['group4']!='PC6-High / SMA-Low']['pid'])
worst_cols = [c for c in rna_log.columns if c in worst_pids]
other_cols = [c for c in rna_log.columns if c in other_pids]

deg_rows = []
for gene in rna_log.index:
    v1 = rna_log.loc[gene, worst_cols].dropna().values.astype(float)
    v2 = rna_log.loc[gene, other_cols].dropna().values.astype(float)
    if len(v1) < 5 or len(v2) < 5:
        continue
    fc  = v1.mean() - v2.mean()
    _, p = ranksums(v1, v2)
    deg_rows.append({'gene': gene, 'logFC': fc, 'p': p})

deg = pd.DataFrame(deg_rows)
_, deg['q'], _, _ = multipletests(deg['p'].values, method='fdr_bh')
deg['sig'] = (deg['q'] < 0.05) & (np.abs(deg['logFC']) >= 0.5)
print(f"  DEG: {deg['sig'].sum()} significant genes")

# ╔══════════════════════════════════════════════════════════════╗
# ║               GSEA 계산 (Hallmark + Immune gene sets)         ║
# ╚══════════════════════════════════════════════════════════════╝

print("Running GSEA...")

HALLMARK_SETS = {
    'HALLMARK_HYPOXIA'                          : ['CA9','NDRG1','VEGFA','HIF1A','ADM','EGLN3','P4HA2','PLOD2','SLC2A1','DDIT4','BNIP3','HMOX1','LDHA','ENO1','PGK1','TGFB1'],
    'HALLMARK_GLYCOLYSIS'                       : ['HK2','PFKP','ALDOA','ENO1','PGK1','PKM','LDHA','GAPDH','TPI1','PGAM1','GPI','ENO2','PFKFB4','LDHB'],
    'HALLMARK_APOPTOSIS'                        : ['CASP3','CASP7','CASP8','CASP9','BCL2','BAX','CYCS','APAF1','TP53'],
    'HALLMARK_INTERFERON_GAMMA_RESPONSE'        : ['STAT1','IRF1','CXCL9','CXCL10','IDO1','CD274','TAP1','B2M'],
    'HALLMARK_INFLAMMATORY_RESPONSE'            : ['IL6','IL1B','TNF','CXCL8','CCL2','PTGS2'],
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB'         : ['NFKB1','RELA','TNF','IL6','CCL2','ICAM1','VCAM1','CXCL1'],
    'HALLMARK_MYC_TARGETS_V1'                  : ['MYC','CDK4','CCNA2','TOP2A','E2F1','NPM1','PCNA'],
    'HALLMARK_ANGIOGENESIS'                     : ['VEGFA','VEGFB','VEGFC','FGF2','ANGPT1','PDGFB'],
    'HALLMARK_E2F_TARGETS'                      : ['E2F1','E2F2','CCNA2','CDK1','MCM2','PCNA','TOP2A'],
    'HALLMARK_TGF_BETA_SIGNALING'               : ['TGFB1','TGFB2','SMAD2','SMAD3','SMAD4'],
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION'        : ['SDHA','SDHB','COX5A','NDUFB3','UQCRC1'],
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION': ['VIM','FN1','SNAI1','TWIST1','ZEB1','CDH2','MMP2'],
    'HALLMARK_WNT_BETA_CATENIN_SIGNALING'       : ['CTNNB1','APC','AXIN1','LRP5','FZD1','TCF7'],
}
IMMUNE_SETS = {
    'CD8_T_CELL'               : ['CD8A','CD8B','GZMA','GZMB','PRF1','IFNG'],
    'TREG'                     : ['FOXP3','IL2RA','IKZF2','CTLA4'],
    'T_CELL_EXHAUSTION'        : ['HAVCR2','PDCD1','LAG3','TOX','TIGIT'],
    'M2_MACROPHAGE'            : ['MSR1','CD163','MRC1','IL10'],
    'M1_MACROPHAGE'            : ['TNF','IL6','IL12A','NOS2','CD80'],
    'MYELOID_IMMUNOSUPPRESSION': ['ARG1','IL10','TGFB1','IDO1'],
    'NK_CELL'                  : ['NCAM1','KLRB1','KLRD1','NKG7','GNLY'],
    'INFLAMMATORY_CYTOKINE'    : ['IL6','IL1B','TNF','CCL2','CXCL8','IL18'],
    'CACHEXIA_SIGNAL'          : ['MSTN','FBXO32','IL6','ACVR2B'],
}

def run_gsea_simple(deg_df, gene_sets):
    scores = deg_df.set_index('gene')['logFC'].to_dict()
    results = []
    for term, genes in gene_sets.items():
        avail = [g for g in genes if g in scores]
        if len(avail) < 3:
            continue
        in_s  = [scores[g] for g in avail]
        out_s = [scores[g] for g in scores if g not in set(genes)]
        if len(out_s) < 10:
            continue
        stat, p = ranksums(in_s, out_s)
        results.append({'Term': term, 'NES': stat, 'FDR q-val': p})
    df = pd.DataFrame(results)
    if len(df) > 0:
        _, df['FDR q-val'], _, _ = multipletests(df['FDR q-val'].values, method='fdr_bh')
    return df

res_h = run_gsea_simple(deg, HALLMARK_SETS)
res_i = run_gsea_simple(deg, IMMUNE_SETS)

# ╔══════════════════════════════════════════════════════════════╗
# ║               Heatmap z-score 계산                            ║
# ╚══════════════════════════════════════════════════════════════╝

print("Computing heatmap z-scores...")

SIGS = {
    'Glycolysis'       : 'HK2,PKM,ALDOA,LDHA,GAPDH,ENO1'.split(','),
    'Hypoxia'          : 'CA9,NDRG1,VEGFA,HIF1A,ADM'.split(','),
    'Lactate/MCT'      : 'LDHB,SLC16A3,SLC16A1'.split(','),
    'CD8 T cell'       : 'CD8A,CD8B,GZMA,GZMB,PRF1'.split(','),
    'T cell Exhaustion': 'HAVCR2,PDCD1,LAG3,TOX'.split(','),
    'Treg'             : 'FOXP3,IL2RA,IKZF2'.split(','),
    'M2 Macrophage'    : 'MSR1,CD163,MRC1'.split(','),
    'IFN-γ Response'   : 'STAT1,IRF1,CXCL9,CXCL10,IDO1'.split(','),
    'Cachexia Signal'  : 'MSTN,FBXO32,IL6,ACVR2B'.split(','),
    'TGF-β'            : 'TGFB1,TGFB2,SMAD2,SMAD3'.split(','),
}

hm_rows = {}
for sig_name, genes in SIGS.items():
    avail = [g for g in genes if g in rna_log.index]
    if not avail:
        continue
    row, all_vals = {}, []
    for grp in G_ORDER:
        pids = [p for p in merged[merged['group4']==grp]['pid'] if p in rna_log.columns]
        val  = rna_log.loc[avail, pids].values.mean()
        row[grp] = val
        all_vals.append(val)
    mu, sd = np.mean(all_vals), np.std(all_vals)
    for grp in G_ORDER:
        row[grp] = (row[grp] - mu) / (sd + 1e-9)
    hm_rows[sig_name] = row

heatmap_z = pd.DataFrame(hm_rows).T   # sigs × groups

SHORT_NAME = {
    'HALLMARK_HYPOXIA':'Hypoxia','HALLMARK_GLYCOLYSIS':'Glycolysis',
    'HALLMARK_APOPTOSIS':'Apoptosis',
    'HALLMARK_INTERFERON_GAMMA_RESPONSE':'IFN-γ Response',
    'HALLMARK_INFLAMMATORY_RESPONSE':'Inflammatory',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB':'TNFα/NF-κB',
    'HALLMARK_MYC_TARGETS_V1':'MYC Targets',
    'HALLMARK_ANGIOGENESIS':'Angiogenesis',
    'HALLMARK_E2F_TARGETS':'E2F Targets',
    'HALLMARK_TGF_BETA_SIGNALING':'TGF-β',
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION':'OXPHOS',
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION':'EMT',
    'HALLMARK_WNT_BETA_CATENIN_SIGNALING':'WNT/β-catenin',
    'CD8_T_CELL':'CD8 T Cell','TREG':'Treg',
    'T_CELL_EXHAUSTION':'T Cell Exhaustion',
    'M2_MACROPHAGE':'M2 Macrophage','M1_MACROPHAGE':'M1 Macrophage',
    'MYELOID_IMMUNOSUPPRESSION':'Myeloid Suppress.',
    'NK_CELL':'NK Cell','INFLAMMATORY_CYTOKINE':'Inflam. Cytokine',
    'CACHEXIA_SIGNAL':'Cachexia Signal',
}

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃                             ║
# ╚══════════════════════════════════════════════════════════════╝

print("Drawing figure...")
fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 6, figure=fig,
              hspace=0.30, wspace=0.38,
              left=0.06, right=0.97, top=0.93, bottom=0.05,
              height_ratios=[1.05, 1.0])
LS, TS, TIS, PS = 11, 9, 11.5, 14

# ── A: KM ─────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0:2])
kmf = KaplanMeierFitter()
for grp in G_ORDER:
    sub = merged[merged['group4']==grp]
    kmf.fit(sub['OS_months'], sub['OS'], label=f"{grp} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax_a, color=G_COLORS[grp], lw=2.2,
                               ci_show=True, ci_alpha=0.10,
                               show_censors=True, censor_styles={'ms':4,'marker':'+'})
lrp = multivariate_logrank_test(merged['OS_months'], merged['group4'], merged['OS']).p_value
ax_a.text(0.97, 0.97, f'Log-rank p={lrp:.2e}',
          transform=ax_a.transAxes, fontsize=8.5, va='top', ha='right',
          bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', alpha=0.9))
ax_a.set_xlabel("Time (months)", fontsize=LS)
ax_a.set_ylabel("Overall Survival Probability", fontsize=LS)
ax_a.set_title('PC6 × SMA 4-Group Survival\n(worst = PC6-High / SMA-Low)', fontsize=TIS, fontweight='bold')
ax_a.set_ylim(-0.08, 1.05)
ax_a.spines[['top','right']].set_visible(False)
ax_a.legend(fontsize=7, loc='upper right').get_frame().set_edgecolor('#ccc')
ax_a.text(-0.10, 1.04, 'A', transform=ax_a.transAxes, fontsize=PS, fontweight='bold')

# ── B: Volcano ────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 2:4])
up = (deg['q']<0.05) & (deg['logFC']>0.5)
dn = (deg['q']<0.05) & (deg['logFC']<-0.5)
ns_mask = ~(up|dn)
ax_b.scatter(deg.loc[ns_mask,'logFC'], -np.log10(deg.loc[ns_mask,'p'].clip(1e-300)),
             c='#CCCCCC', s=3, alpha=0.4, rasterized=True)
ax_b.scatter(deg.loc[up,'logFC'], -np.log10(deg.loc[up,'p'].clip(1e-300)),
             c='#D73027', s=8, alpha=0.75, rasterized=True, label=f'Up ({up.sum()})')
ax_b.scatter(deg.loc[dn,'logFC'], -np.log10(deg.loc[dn,'p'].clip(1e-300)),
             c='#2166AC', s=8, alpha=0.75, rasterized=True, label=f'Down ({dn.sum()})')
for g, col in [('CA9','#D73027'),('NDRG1','#D73027'),('SLC16A3','#D73027'),
               ('VEGFA','#D73027'),('ALDOA','#D73027'),('PKM','#D73027'),
               ('HAVCR2','#D73027'),('MSTN','#D73027'),('MRC1','#2166AC')]:
    row = deg[deg['gene']==g]
    if len(row)==0: continue
    r = row.iloc[0]
    ax_b.annotate(g, (r['logFC'], -np.log10(r['p']+1e-300)),
                  xytext=(4,2), textcoords='offset points',
                  fontsize=7.5, fontweight='bold', color=col)
foxp3 = deg[deg['gene']=='FOXP3']
if len(foxp3) > 0:
    fx = foxp3.iloc[0]['logFC']; fy = -np.log10(foxp3.iloc[0]['p']+1e-300)
    ax_b.scatter(fx, fy, c='#2166AC', s=60, zorder=5, edgecolors='white', linewidths=1.2)
    ax_b.annotate('FOXP3', xy=(fx, fy), xytext=(fx-0.55, fy+1.2),
                  fontsize=9, fontweight='bold', color='#2166AC',
                  bbox=dict(boxstyle='round,pad=0.25', fc='#EBF5FB', ec='#2166AC', alpha=0.92),
                  arrowprops=dict(arrowstyle='->', color='#2166AC', lw=1.5))
ax_b.axvline(0.5, color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.axvline(-0.5, color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.axhline(-np.log10(0.05), color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.set_xlabel("log₂FC (PC6-High/SMA-Low vs Others)", fontsize=LS)
ax_b.set_ylabel("-log₁₀(p)", fontsize=LS)
ax_b.set_title('DEG: PC6-High/SMA-Low vs All Others\n(FDR<0.05, |log₂FC|≥0.5)', fontsize=TIS, fontweight='bold')
ax_b.legend(fontsize=9, markerscale=2).get_frame().set_edgecolor('#ccc')
ax_b.spines[['top','right']].set_visible(False)
ax_b.text(-0.10, 1.04, 'B', transform=ax_b.transAxes, fontsize=PS, fontweight='bold')

# ── C: GSEA ───────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 4:6])
gsea_comb = pd.concat([res_h, res_i], ignore_index=True).sort_values('NES')
gsea_comb['label'] = gsea_comb['Term'].map(SHORT_NAME).fillna(gsea_comb['Term'])
cols_c = ['#D73027' if (n>0 and q<0.05) else '#F4A582' if n>0
          else '#2166AC' if q<0.05 else '#92C5DE'
          for n,q in zip(gsea_comb['NES'], gsea_comb['FDR q-val'])]
ax_c.barh(range(len(gsea_comb)), gsea_comb['NES'], color=cols_c, alpha=0.88, edgecolor='white')
ax_c.set_yticks(range(len(gsea_comb)))
ax_c.set_yticklabels(gsea_comb['label'], fontsize=8.5)
ax_c.axvline(0, color='black', lw=0.8)
for i, (_, row) in enumerate(gsea_comb.iterrows()):
    q = row['FDR q-val']
    sig = '***' if q<0.001 else '**' if q<0.01 else '*' if q<0.05 else ''
    if sig:
        x = row['NES'] + (0.04 if row['NES']>0 else -0.04)
        ax_c.text(x, i, sig, va='center', ha='left' if row['NES']>0 else 'right',
                  fontsize=9, color='#333', fontweight='bold')
ax_c.set_xlabel("NES", fontsize=LS)
ax_c.set_title('Pathway Enrichment (GSEA)\nPC6-High/SMA-Low vs Others', fontsize=TIS, fontweight='bold')
ax_c.spines[['top','right']].set_visible(False)
sig_p = mpatches.Patch(color='#D73027', label='Enriched FDR<0.05')
ns_p  = mpatches.Patch(color='#F4A582', label='Enriched ns')
dep_p = mpatches.Patch(color='#2166AC', label='Depleted FDR<0.05')
ax_c.legend(handles=[sig_p,ns_p,dep_p], fontsize=7.5, loc='lower right').get_frame().set_edgecolor('#ccc')
ax_c.text(-0.10, 1.04, 'C', transform=ax_c.transAxes, fontsize=PS, fontweight='bold')

# ── D: Heatmap ────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1:3])
hm = heatmap_z[G_ORDER]
im = ax_d.imshow(hm.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax_d.set_xticks(range(4))
ax_d.set_xticklabels(G_LABELS, fontsize=8.5)
ax_d.set_yticks(range(len(hm)))
ax_d.set_yticklabels(hm.index, fontsize=9)
for i, grp in enumerate(G_ORDER):
    ax_d.get_xticklabels()[i].set_color(G_COLORS[grp])
    ax_d.get_xticklabels()[i].set_fontweight('bold')
plt.colorbar(im, ax=ax_d, shrink=0.75, pad=0.02, label='z-score')
ax_d.set_title('Transcriptional Signature Heatmap\n(z-score across 4 groups)', fontsize=TIS, fontweight='bold')
for si, sig_name in enumerate(hm.index):
    genes = SIGS.get(sig_name, [])
    avail = [g for g in genes if g in rna_log.index]
    if not avail: continue
    gv = [rna_log.loc[avail,
          [p for p in merged[merged['group4']==g]['pid'] if p in rna_log.columns]
          ].values.flatten() for g in G_ORDER]
    try:
        _, kp = kruskal(*gv)
        sig = '***' if kp<0.001 else '**' if kp<0.01 else '*' if kp<0.05 else ''
        if sig:
            ax_d.text(3.65, si, sig, va='center', ha='left', fontsize=8, color='#333', fontweight='bold')
    except:
        pass
ax_d.text(-0.15, 1.04, 'D', transform=ax_d.transAxes, fontsize=PS, fontweight='bold')

# ── E: Violin ─────────────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 3:5])
markers = [('LDHA','Glycolysis','#E74C3C'),
           ('SLC16A3','MCT4','#C0392B'),
           ('HAVCR2','T Exhaustion','#8E44AD'),
           ('FOXP3','Treg','#27AE60'),
           ('MSTN','Cachexia','#7F8C8D')]
group_x_base = np.arange(len(markers)) * 5.5
for mi, (gene, cat, col) in enumerate(markers):
    if gene not in rna_log.index: continue
    for gi, grp in enumerate(G_ORDER):
        pids = [p for p in merged[merged['group4']==grp]['pid'] if p in rna_log.columns]
        vals = rna_log.loc[gene, pids].values
        xpos = group_x_base[mi] + gi
        vp = ax_e.violinplot([vals], positions=[xpos], widths=0.8,
                              showmedians=True, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor(G_COLORS[grp])
            body.set_alpha(0.80 if grp=='PC6-High / SMA-Low' else 0.42)
            body.set_edgecolor('white')
        vp['cmedians'].set_color(G_COLORS[grp]); vp['cmedians'].set_linewidth(2)
    groups_vals = [rna_log.loc[gene,
                   [p for p in merged[merged['group4']==g]['pid'] if p in rna_log.columns]
                   ].values for g in G_ORDER]
    _, kp = kruskal(*groups_vals)
    sig = '***' if kp<0.001 else '**' if kp<0.01 else '*' if kp<0.05 else 'ns'
    ymax = max(v.max() for v in groups_vals)
    ax_e.text(group_x_base[mi]+1.5, ymax+0.15, sig,
              ha='center', fontsize=11, fontweight='bold',
              color='#c0392b' if sig!='ns' else '#888')
    ax_e.text(group_x_base[mi]+1.5, -0.7,
              f'{gene}\n({cat})', ha='center', fontsize=8.5, fontweight='bold', color=col)
ax_e.set_ylabel("log₂(TPM+1)", fontsize=LS)
ax_e.set_title('Representative Marker Expression\nby PC6 × SMA 4 Groups', fontsize=TIS, fontweight='bold')
ax_e.set_xlim(-0.8, group_x_base[-1]+3.8)
ax_e.set_xticks([])
ax_e.spines[['top','right']].set_visible(False)
handles_e = [mpatches.Patch(color=G_COLORS[g], label=g, alpha=0.75) for g in G_ORDER]
ax_e.legend(handles=handles_e, fontsize=8, loc='upper left', ncol=2).get_frame().set_edgecolor('#ccc')
ax_e.text(-0.10, 1.04, 'E', transform=ax_e.transAxes, fontsize=PS, fontweight='bold')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle('Joint Effects of PC6 and Body Composition',
             fontsize=13.5, fontweight='bold', y=0.98)

plt.savefig(OUT_TIFF, dpi=300, format='tiff', bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Loading data...
Computing DEG...
  DEG: 824 significant genes
Running GSEA...
Computing heatmap z-scores...
Drawing figure...

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure6.tiff


In [27]:
"""
Figure 4. Clinical Impact of PC6 in Combination with Body Composition
=====================================================================
(A) TCGA:   multivariable Cox (SMA vs PC6)
(B) CPTAC:  multivariable Cox (SMA vs PC6)
(C) Risk map (SMA × PC6, TCGA-centered)

수정사항:
  - A, B 변수 라벨: axes 왼쪽 바깥 고정 (CI선과 겹침 없음)
  - HR 텍스트: 점 바로 아래 배치
  - 제목 + cohort info 통합

Input
-----
  TCGA_with_genes.csv
  External_with_genes.csv
  ae_latent_pca10_all.csv
  ae_latent_pca10_external.csv

Output
------
  Figure4_fixed.tiff

Run
---
  python figure4_fixed.py
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
BASE = r"C:\Users\nsw\Desktop\body_composition"

TCGA_PATH = os.path.join(BASE, "TCGA_with_genes.csv")
EXT_PATH  = os.path.join(BASE, "External_with_genes.csv")
AE_TCGA   = os.path.join(BASE, "ae_latent_pca10_all.csv")
AE_EXT    = os.path.join(BASE, "ae_latent_pca10_external.csv")
OUTPATH   = os.path.join(BASE, "Figure4_fixed.tiff")

SELECTED_PC = "PC6"


# ============================================================
# HELPERS
# ============================================================

def star(p):
    if pd.isna(p): return "na"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


def fmt_p(p):
    if pd.isna(p): return "NA"
    if p < 0.001:  return "<0.001"
    return f"{p:.3f}"


def format_stage_binary(x):
    s = str(x).upper()
    if ("III" in s) or ("IV" in s): return 1.0
    if ("II"  in s) or ("I"  in s): return 0.0
    return np.nan


def pretty_var_name(v, selected_pc):
    if v == "SMA_zscore":           return "SMA z-score"
    if v == f"AE_{selected_pc}_z": return selected_pc
    return v


def preprocess_and_merge(clin_df, ae_df, cohort_name):
    df = clin_df.copy()
    ae = ae_df.copy()

    df["age"]        = pd.to_numeric(df["age"],        errors="coerce")
    df["OS"]         = pd.to_numeric(df["OS"],         errors="coerce")
    df["OS_time"]    = pd.to_numeric(df["OS_time"],    errors="coerce") / 30.44
    df["SMA_zscore"] = pd.to_numeric(df["SMA_zscore"], errors="coerce")
    df["sex_bin"]    = (df["sex"].astype(str).str.upper() == "MALE").astype(float)
    df["stage_bin"]  = df["stage"].apply(format_stage_binary)

    if cohort_name == "TCGA":
        df["pid"] = df["patient_id"].astype(str).str[:12]
        ae["pid"] = ae["patient_id"].astype(str).str[:12]
    else:
        df["pid"] = df["patient_id"].astype(str)
        ae["pid"] = ae["patient_id"].astype(str)

    use_cols = ["pid"] + [f"AE_PC{i}" for i in range(1, 11)]
    merged = df.merge(ae[use_cols], on="pid", how="inner")

    for i in range(1, 11):
        col  = f"AE_PC{i}"
        zcol = f"{col}_z"
        std  = merged[col].std()
        if pd.isna(std) or std == 0:
            merged[zcol] = np.nan
        else:
            merged[zcol] = (merged[col] - merged[col].mean()) / std

    return merged


def fit_joint_model(df, selected_pc):
    pc_col   = f"AE_{selected_pc}_z"
    use_cols = [
        "OS_time", "OS", "cancer_type",
        "age", "sex_bin", "stage_bin",
        "SMA_zscore", pc_col
    ]
    tmp = df[use_cols].dropna().copy()
    tmp = tmp[tmp["OS_time"] > 0].copy()

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(tmp, duration_col="OS_time", event_col="OS", strata="cancer_type")

    out_rows = []
    for var in ["SMA_zscore", pc_col]:
        s = cph.summary.loc[var]
        out_rows.append({
            "variable": var,
            "label"   : pretty_var_name(var, selected_pc),
            "hr"      : float(s["exp(coef)"]),
            "lo"      : float(s["exp(coef) lower 95%"]),
            "hi"      : float(s["exp(coef) upper 95%"]),
            "p"       : float(s["p"]),
        })
    return cph, tmp, pd.DataFrame(out_rows)


# ════════════════════════════════════════════════════════════════
# Forest plot (A, B 공통)
# ════════════════════════════════════════════════════════════════

def draw_joint_forest(ax, result_df, cohort_label, n, events, xlim, selected_pc):
    plot_df = result_df.copy()
    order   = ["SMA z-score", selected_pc]
    plot_df["order"] = plot_df["label"].map({k: i for i, k in enumerate(order)})
    plot_df = plot_df.sort_values("order").reset_index(drop=True)

    y_positions = np.array([1, 0])
    ax.axvline(1.0, color="#777777", lw=1.1, ls="--", zorder=0)

    for i, row in plot_df.iterrows():
        y      = y_positions[i]
        is_pc  = row["label"] == selected_pc
        color  = "#C0392B" if is_pc else "#2E86C1"
        marker = "s"       if is_pc else "D"
        fc     = "#FDEDEC" if is_pc else "#EBF5FB"
        alpha  = 0.85      if is_pc else 0.75

        ax.axhspan(y - 0.42, y + 0.42, color=fc, alpha=alpha, zorder=0)

        ax.errorbar(
            row["hr"], y,
            xerr=[[row["hr"] - row["lo"]], [row["hi"] - row["hr"]]],
            fmt=marker, color=color,
            markerfacecolor=color, markeredgecolor="#111111",
            markeredgewidth=0.9,
            markersize=8.5 if row["p"] < 0.05 else 6.5,
            elinewidth=1.8 if row["p"] < 0.05 else 1.2,
            capsize=3.5, zorder=3
        )

        # ── 변수 라벨: axes 왼쪽 바깥 고정 (CI선과 겹치지 않음) ──
        ax.text(-0.18, y, row["label"],
                transform=ax.get_yaxis_transform(),
                ha="left", va="center", fontsize=10.5,
                fontweight="bold", color=color)

        # ── HR 텍스트: 점 바로 아래 (CI선과 겹치지 않음) ─────────
        hr_text = (f"HR={row['hr']:.2f} ({row['lo']:.2f}–{row['hi']:.2f})\n"
                   f"p={fmt_p(row['p'])} {star(row['p'])}")
        ax.text(row["hr"], y - 0.28, hr_text,
                ha="center", va="top", fontsize=8.0,
                color=color,
                fontweight="bold" if row["p"] < 0.05 else "normal")

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(-0.85, 1.75)
    ax.set_yticks([])
    ax.set_xlabel("Hazard Ratio (95% CI)", fontsize=11)

    # ── 제목 + cohort info 통합 ────────────────────────────────
    ax.set_title(
        f"{cohort_label}\n"
        f"n={n}, events={events} | adj: age, sex, stage | strat: cancer type",
        fontsize=10.5, fontweight="bold", pad=12, linespacing=1.5
    )

    legend_handles = [
        Line2D([0],[0], marker="D", color="w", markerfacecolor="#2E86C1",
               markeredgecolor="#111111", markersize=7.5, label="SMA z-score"),
        Line2D([0],[0], marker="s", color="w", markerfacecolor="#C0392B",
               markeredgecolor="#111111", markersize=7.5, label=selected_pc),
        mpatches.Patch(facecolor="#EBF5FB", edgecolor="#2E86C1", label="Body composition"),
        mpatches.Patch(facecolor="#FDEDEC", edgecolor="#C0392B", label="Latent feature"),
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=8, framealpha=0.95)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)


# ════════════════════════════════════════════════════════════════
# Risk surface (C)
# ════════════════════════════════════════════════════════════════

def build_risk_map(ax, cph_model, tcga_df, selected_pc):
    pc_col   = f"AE_{selected_pc}_z"
    beta_sma = float(cph_model.params_["SMA_zscore"])
    beta_pc  = float(cph_model.params_[pc_col])

    x_min = np.nanquantile(tcga_df["SMA_zscore"], 0.01)
    x_max = np.nanquantile(tcga_df["SMA_zscore"], 0.99)
    y_min = np.nanquantile(tcga_df[pc_col], 0.01)
    y_max = np.nanquantile(tcga_df[pc_col], 0.99)

    x = np.linspace(x_min, x_max, 220)
    y = np.linspace(y_min, y_max, 220)
    X, Y = np.meshgrid(x, y)
    RR   = np.exp(beta_sma * X + beta_pc * Y)

    vmin = np.nanquantile(RR, 0.02)
    vmax = np.nanquantile(RR, 0.98)

    cf = ax.contourf(X, Y, RR, levels=14, cmap="RdYlBu_r",
                     vmin=vmin, vmax=vmax, alpha=0.95)
    cs = ax.contour(X, Y, RR, levels=8, colors="white",
                    linewidths=0.6, alpha=0.55)
    ax.clabel(cs, inline=True, fontsize=6.5, fmt="%.2f")

    ax.scatter(tcga_df["SMA_zscore"], tcga_df[pc_col],
               s=10, facecolors="white", edgecolors="#999999",
               linewidths=0.25, alpha=0.45, zorder=3)

    ax.axvline(0, color="#444444", lw=0.9, ls="--", alpha=0.8)
    ax.axhline(0, color="#444444", lw=0.9, ls="--", alpha=0.8)

    ax.set_xlabel("SMA z-score",          fontsize=10.8)
    ax.set_ylabel(f"{selected_pc} z-score", fontsize=10.8)
    ax.set_title("(C) TCGA-centered risk map\nCombined effect of SMA and PC6",
                 fontsize=11.0, fontweight="bold", pad=16)

    ax.text(0.03, 0.99,
            "Relative hazard from TCGA joint Cox model\n"
            r"$RR = \exp(\beta_{SMA}\cdot SMA_z + \beta_{PC6}\cdot PC6_z)$",
            transform=ax.transAxes, ha="left", va="top", fontsize=7.6,
            bbox=dict(boxstyle="round,pad=0.28", facecolor="white",
                      edgecolor="#BBBBBB", alpha=0.92))

    corners = [
        {"name": "Low SMA\nHigh PC6",  "x": x_min, "y": y_max},
        {"name": "High SMA\nHigh PC6", "x": x_max, "y": y_max},
        {"name": "Low SMA\nLow PC6",   "x": x_min, "y": y_min},
        {"name": "High SMA\nLow PC6",  "x": x_max, "y": y_min},
    ]
    for c in corners:
        c["rr"] = np.exp(beta_sma * c["x"] + beta_pc * c["y"])

    highest = max(corners, key=lambda z: z["rr"])
    lowest  = min(corners, key=lambda z: z["rr"])
    x_pad   = 0.06 * (x_max - x_min)
    y_pad   = 0.06 * (y_max - y_min)

    ax.text(highest["x"] + x_pad, highest["y"] - y_pad,
            f"Highest risk\n{highest['name']}",
            ha="center", va="center", fontsize=8.3, fontweight="bold",
            color="#7B241C",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="#FDEDEC",
                      edgecolor="#C0392B", alpha=0.95))

    ax.text(lowest["x"] - x_pad, lowest["y"] + y_pad,
            f"Lowest risk\n{lowest['name']}",
            ha="center", va="center", fontsize=8.3, fontweight="bold",
            color="#1B4F72",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="#EBF5FB",
                      edgecolor="#2E86C1", alpha=0.95))

    cbar = plt.colorbar(cf, ax=ax, fraction=0.050, pad=0.03)
    cbar.set_label("Relative hazard", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


# ============================================================
# LOAD DATA
# ============================================================
print("Loading data...")
tcga = pd.read_csv(TCGA_PATH)
ext  = pd.read_csv(EXT_PATH)
ae_t = pd.read_csv(AE_TCGA)
ae_e = pd.read_csv(AE_EXT)

df_tcga = preprocess_and_merge(tcga, ae_t, cohort_name="TCGA")
df_ext  = preprocess_and_merge(ext,  ae_e, cohort_name="CPTAC")
print(f"  TCGA merged n = {len(df_tcga)}")
print(f"  CPTAC merged n = {len(df_ext)}")

# ============================================================
# FIT JOINT MODELS
# ============================================================
print("Fitting joint Cox models...")
tcga_model, tcga_used, tcga_res = fit_joint_model(df_tcga, SELECTED_PC)
ext_model,  ext_used,  ext_res  = fit_joint_model(df_ext,  SELECTED_PC)

print("\n[TCGA joint model]")
for _, row in tcga_res.iterrows():
    print(f"  {row['label']}: HR={row['hr']:.3f} ({row['lo']:.3f}-{row['hi']:.3f})  p={row['p']:.4f}")

print("\n[CPTAC joint model]")
for _, row in ext_res.iterrows():
    print(f"  {row['label']}: HR={row['hr']:.3f} ({row['lo']:.3f}-{row['hi']:.3f})  p={row['p']:.4f}")

# ============================================================
# COMMON X LIMIT
# ============================================================
all_lo = pd.concat([tcga_res["lo"], ext_res["lo"]]).replace([np.inf,-np.inf], np.nan).dropna()
all_hi = pd.concat([tcga_res["hi"], ext_res["hi"]]).replace([np.inf,-np.inf], np.nan).dropna()
xmin = max(0.35, float(all_lo.min()) * 0.88)
xmax = max(2.15, float(all_hi.max()) * 1.22)
xlim = (xmin, xmax)

# ============================================================
# PLOT
# ============================================================
fig = plt.figure(figsize=(24, 9))
fig.patch.set_facecolor("white")
gs = GridSpec(1, 3, figure=fig,
              width_ratios=[1.05, 1.05, 1.12],
              wspace=0.55,
              left=0.10, right=0.97,
              top=0.84, bottom=0.12)

# Panel A
ax1 = fig.add_subplot(gs[0, 0])
draw_joint_forest(
    ax=ax1,
    result_df=tcga_res,
    cohort_label="(A) TCGA: multivariable Cox model",
    n=len(tcga_used),
    events=int(tcga_used["OS"].sum()),
    xlim=xlim,
    selected_pc=SELECTED_PC
)

# Panel B
ax2 = fig.add_subplot(gs[0, 1])
draw_joint_forest(
    ax=ax2,
    result_df=ext_res,
    cohort_label="(B) CPTAC-3: multivariable Cox model",
    n=len(ext_used),
    events=int(ext_used["OS"].sum()),
    xlim=xlim,
    selected_pc=SELECTED_PC
)

# Panel C
ax3 = fig.add_subplot(gs[0, 2])
build_risk_map(
    ax=ax3,
    cph_model=tcga_model,
    tcga_df=tcga_used,
    selected_pc=SELECTED_PC
)

# ============================================================
# TITLE
# ============================================================
fig.suptitle(
    "Figure 4.  Clinical Impact of PC6 in Combination with Body Composition",
    fontsize=15, fontweight="bold", y=0.97
)

plt.savefig(OUTPATH, dpi=300, bbox_inches="tight", facecolor="white", format="tiff")
plt.close()

print(f"\nSaved: {OUTPATH}")

Loading data...
  TCGA merged n = 495
  CPTAC merged n = 179
Fitting joint Cox models...

[TCGA joint model]
  SMA z-score: HR=0.762 (0.655-0.886)  p=0.0004
  PC6: HR=1.170 (0.998-1.372)  p=0.0535

[CPTAC joint model]
  SMA z-score: HR=0.837 (0.636-1.102)  p=0.2054
  PC6: HR=1.404 (1.078-1.828)  p=0.0118

Saved: C:\Users\nsw\Desktop\body_composition\Figure4_fixed.tiff


In [42]:
"""
=============================================================================
Biological Characterization of All 10 Latent Components
=============================================================================
1. Extract encoder weights → gene-to-PC weight for ALL 10 PCs
2. Pathway enrichment (Hallmark, Wilcoxon rank-sum) for ALL 10 PCs
3. Summary heatmap: PC × Pathway
4. Detailed volcano/bar for prognostically significant PC
5. Correlation validation
=============================================================================
"""

import pandas as pd
import numpy as np
import warnings
import torch
from scipy.stats import spearmanr, ranksums
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD AUTOENCODER MODEL
# =============================================================================
# ★★ 경로를 본인 환경에 맞게 수정하세요 ★★
bundle = torch.load(r'C:\Users\nsw\Desktop\body_composition\ae_v3_outputs\ae_model_bundle.pt',
                    map_location='cpu', weights_only=False)

state_dict = bundle['state_dict']
gene_names = bundle['selected_genes']
pca_components = bundle['pca10_components']  # (10, 64)

# Encoder: Linear(0) → BN(1) → ReLU → Dropout → Linear(4) → BN(5) → ReLU → Dropout → Linear(8)
W1 = state_dict['encoder.0.weight'].numpy()   # (1024, 3000)
W2 = state_dict['encoder.4.weight'].numpy()   # (256, 1024)
W3 = state_dict['encoder.8.weight'].numpy()   # (64, 256)

print(f"Genes: {len(gene_names)}")
print(f"Encoder: {W1.shape[1]} → {W1.shape[0]} → {W2.shape[0]} → {W3.shape[0]} → PCA(10)")

# =============================================================================
# 2. GENE → PC WEIGHT FOR ALL 10 PCs
# =============================================================================
all_pc_weights = {}

for pc_idx in range(10):
    pc_loading = pca_components[pc_idx, :]
    weights = pc_loading @ W3 @ W2 @ W1
    all_pc_weights[f'PC{pc_idx+1}'] = weights

# Save full weight matrix
weight_df = pd.DataFrame(all_pc_weights, index=gene_names)
weight_df.index.name = 'gene'
weight_df.to_csv('gene_weights_all_PCs.csv')
print(f"\n✓ Gene weights for all 10 PCs saved: gene_weights_all_PCs.csv")

# Top genes per PC
print(f"\n{'='*80}")
print("TOP 5 RISK / PROTECTIVE GENES PER PC")
print(f"{'='*80}")
for pc in [f'PC{i}' for i in range(1, 11)]:
    w = weight_df[pc]
    top_risk = w.nlargest(5)
    top_prot = w.nsmallest(5)
    risk_str = ', '.join(f'{g}({v:+.2f})' for g, v in top_risk.items())
    prot_str = ', '.join(f'{g}({v:+.2f})' for g, v in top_prot.items())
    print(f"\n  {pc}:")
    print(f"    Risk↑:  {risk_str}")
    print(f"    Prot↓:  {prot_str}")

# =============================================================================
# 3. HALLMARK GENE SETS
# =============================================================================
hallmark_sets = {
    'Glycolysis': ['PKM','ENO1','SLC16A3','PFKP','PGAM1','LDHA','AKR1B1','GAPDH',
                   'GPI','TPI1','ALDOA','PGK1','HK2','HK1','PFKL','ENO2','LDHB',
                   'SLC2A1','SLC2A3','BPGM','PCK1','PCK2','FBP1','ALDOB','ALDOC'],
    'Hypoxia': ['NDRG1','VEGFA','LOX','BNIP3','ANGPTL4','SLC2A1','ADM','P4HA1',
                'DDIT4','EGLN3','PDK1','LDHA','PGK1','HK2','ENO1','ALDOA','SLC2A3',
                'CA9','BNIP3L','IGFBP3','ANKRD37','PLOD2','MXI1','ERO1A','PFKFB3'],
    'IFN-alpha Response': ['ISG15','MX1','IFIT1','IFIT3','OAS1','OAS2','OAS3','STAT1',
                           'IRF7','IFI44','IFI44L','RSAD2','IFI6','HERC5','USP18',
                           'IFITM1','BST2','XAF1','TRIM22','DDX58','CXCL10','GBP1'],
    'IFN-gamma Response': ['GBP1','STAT1','IRF1','CXCL9','CXCL10','CXCL11','IDO1',
                           'TAP1','TAP2','PSMB9','PSMB8','HLA-A','HLA-B','HLA-C',
                           'B2M','CD274','CIITA','GBP2','GBP4','GBP5','WARS1'],
    'Cholesterol Homeostasis': ['PCSK9','INSIG1','HMGCR','HMGCS1','FDFT1','SQLE',
                                 'MVD','MVK','IDI1','FDPS','LSS','DHCR7','DHCR24',
                                 'SC5D','CYP51A1','MSMO1','NSDHL','HSD17B7','EBP',
                                 'LDLR','SREBF2','ACAT2','ACLY','SCD'],
    'mTORC1 Signaling': ['SLC7A5','SLC3A2','SLC1A5','MTHFD2','SHMT2','PSAT1','PHGDH',
                          'PSPH','CBS','CTH','SRM','ODC1','AMD1','MTOR','RPTOR',
                          'EIF4EBP1','RPS6KB1','DDIT4','SESN2','TRIB3','NUPR1'],
    'p53 Pathway': ['CDKN1A','MDM2','BAX','BBC3','PMAIP1','GADD45A','FAS','TNFRSF10B',
                    'TP53INP1','SESN1','SESN2','DDIT4','PLK2','PLK3','CCNG1','DDB2',
                    'XPC','FDXR','RRM2B','TIGAR','ZMAT3','PHLDA3'],
    'TNF-alpha/NF-kB': ['TNFAIP3','NFKBIA','NFKB1','RELA','RELB','BCL3','CXCL1',
                         'CXCL2','CXCL3','CCL2','CCL5','IL6','IL8','ICAM1','VCAM1',
                         'TRAF1','BIRC3','BIRC2','CFLAR','TNFAIP2','TNFAIP8'],
    'EMT': ['VIM','CDH2','SNAI1','SNAI2','TWIST1','ZEB1','ZEB2','FN1',
            'SPARC','MMP2','MMP9','COL1A1','COL1A2','COL3A1','COL5A1',
            'ACTA2','TAGLN','POSTN','LOX','LOXL2','SERPINE1','ITGA5'],
    'Angiogenesis': ['VEGFA','VEGFB','VEGFC','FLT1','KDR','FLT4','ANGPT1','ANGPT2',
                     'TEK','PECAM1','CDH5','ENG','PDGFB','PDGFRB','NRP1','NRP2',
                     'HIF1A','EPAS1','DLL4','NOTCH1','JAG1','THBS1'],
    'Inflammatory Response': ['IL6','IL1B','IL1A','TNF','CXCL8','CCL2','CCL3','CCL4',
                               'CXCL1','CXCL2','CXCL3','PTGS2','SELE','ICAM1','VCAM1',
                               'NFKB1','TLR2','TLR4','MYD88','IRAK4','TRAF6'],
    'Bile Acid Metabolism': ['CYP7A1','CYP8B1','CYP27A1','BAAT','SLC27A5','ABCB11',
                              'NR1H4','SLCO1B1','SLCO1B3','SLC10A1','SLC10A2',
                              'AKR1D1','HSD3B7','AMACR','ACOX2','SCP2','UGT2B4'],
    'Xenobiotic Metabolism': ['CYP1A1','CYP1A2','CYP1B1','CYP2B6','CYP2C9','CYP2C19',
                               'CYP2D6','CYP2E1','CYP3A4','CYP3A5','UGT1A1','UGT1A6',
                               'GSTA1','GSTA2','GSTM1','GSTP1','GSTT1','NR1I2','NR1I3',
                               'EPHX1','ALDH3A1','AKR1C1','AKR1C2','AKR1C3'],
    'Fatty Acid Metabolism': ['ACACA','FASN','SCD','ELOVL6','FADS1','FADS2','ACSL1',
                               'ACSL3','ACSL4','ACSL5','CPT1A','CPT2','ACADM','ACADL',
                               'ACADVL','HADHA','HADHB','ECHS1','EHHADH','ACOX1'],
    'Oxidative Phosphorylation': ['NDUFA1','NDUFA2','NDUFB1','NDUFS1','NDUFS2',
                                   'SDHA','SDHB','UQCRC1','UQCRC2','CYC1',
                                   'COX5A','COX5B','COX6A1','COX7A2','ATP5F1A',
                                   'ATP5F1B','ATP5F1C','ATP5MC1','ATP5PD'],
    'Apoptosis': ['BAX','BAK1','BID','BIM','BAD','CASP3','CASP7','CASP8','CASP9',
                  'CYCS','APAF1','DIABLO','XIAP','BCL2','BCL2L1','MCL1','TNFRSF10A',
                  'TNFRSF10B','FAS','FASLG','TRAIL'],
    'DNA Repair': ['BRCA1','BRCA2','RAD51','ATM','ATR','CHEK1','CHEK2','TP53',
                   'XRCC1','PARP1','MLH1','MSH2','MSH6','PMS2','ERCC1','ERCC2',
                   'XPA','XPC','OGG1','MUTYH','MGMT'],
    'Cell Cycle (G2M)': ['CDK1','CCNB1','CCNB2','CCNA2','CDC20','CDC25C','PLK1',
                          'AURKA','AURKB','BUB1','BUB1B','MAD2L1','TTK','NDC80',
                          'CENPA','CENPF','TOP2A','MKI67','KIF11','KIF23','BIRC5'],
    'Complement': ['C1QA','C1QB','C1QC','C2','C3','C4A','C4B','C5','C6','C7',
                   'C8A','C8B','C9','CFB','CFD','CFH','CFI','MASP1','MASP2','MBL2'],
}

# =============================================================================
# 4. PATHWAY ENRICHMENT FOR ALL 10 PCs
# =============================================================================
print(f"\n{'='*90}")
print("PATHWAY ENRICHMENT — ALL 10 PCs (Wilcoxon rank-sum)")
print(f"{'='*90}")

weight_dict_all = {}
for pc in [f'PC{i}' for i in range(1, 11)]:
    weight_dict_all[pc] = dict(zip(gene_names, all_pc_weights[pc]))

all_enrich = []
for pc in [f'PC{i}' for i in range(1, 11)]:
    wd = weight_dict_all[pc]
    for pathway, genes in hallmark_sets.items():
        present = [g for g in genes if g in wd]
        if len(present) < 5:
            continue
        in_w = np.array([wd[g] for g in present])
        out_w = np.array([wd[g] for g in gene_names if g not in present])
        stat, pval = ranksums(in_w, out_w)
        all_enrich.append({
            'PC': pc, 'Pathway': pathway,
            'N_genes': len(present), 'Mean_weight': in_w.mean(),
            'Statistic': stat, 'p_value': pval,
        })

enrich_df = pd.DataFrame(all_enrich)

# FDR per PC
fdr_list = []
for pc in [f'PC{i}' for i in range(1, 11)]:
    sub = enrich_df[enrich_df['PC'] == pc].copy()
    _, fdr, _, _ = multipletests(sub['p_value'], method='fdr_bh')
    sub['FDR'] = fdr
    fdr_list.append(sub)
enrich_df = pd.concat(fdr_list, ignore_index=True)

enrich_df.to_csv('pathway_enrichment_all_PCs.csv', index=False)
print(f"✓ Saved: pathway_enrichment_all_PCs.csv")

# Print significant per PC
for pc in [f'PC{i}' for i in range(1, 11)]:
    sub = enrich_df[(enrich_df['PC'] == pc) & (enrich_df['FDR'] < 0.05)]
    sub = sub.sort_values('Mean_weight', ascending=False)
    if len(sub) == 0:
        print(f"\n  {pc}: No pathways FDR<0.05")
    else:
        print(f"\n  {pc}: {len(sub)} pathways FDR<0.05")
        for _, r in sub.iterrows():
            direction = '↑Risk' if r['Mean_weight'] > 0 else '↓Protective'
            print(f"    {r['Pathway']:30s} mean_w={r['Mean_weight']:+.3f} FDR={r['FDR']:.4f} {direction}")

# =============================================================================
# 5. HEATMAP: PC × Pathway (signed -log10 FDR)
# =============================================================================
pathways_order = sorted(hallmark_sets.keys())
pcs_order = [f'PC{i}' for i in range(1, 11)]

heatmap_data = np.zeros((len(pathways_order), len(pcs_order)))
for i, pathway in enumerate(pathways_order):
    for j, pc in enumerate(pcs_order):
        row = enrich_df[(enrich_df['PC'] == pc) & (enrich_df['Pathway'] == pathway)]
        if len(row) == 0:
            heatmap_data[i, j] = 0
        else:
            fdr = row.iloc[0]['FDR']
            mean_w = row.iloc[0]['Mean_weight']
            val = -np.log10(max(fdr, 1e-10))
            heatmap_data[i, j] = val if mean_w > 0 else -val

fig, ax = plt.subplots(figsize=(14, 10))

vmax = np.abs(heatmap_data).max()
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(heatmap_data, cmap='RdBu_r', norm=norm, aspect='auto')

ax.set_xticks(range(len(pcs_order)))
ax.set_xticklabels(pcs_order, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(pathways_order)))
ax.set_yticklabels(pathways_order, fontsize=9)

for i in range(len(pathways_order)):
    for j in range(len(pcs_order)):
        row = enrich_df[(enrich_df['PC'] == pcs_order[j]) &
                        (enrich_df['Pathway'] == pathways_order[i])]
        if len(row) > 0 and row.iloc[0]['FDR'] < 0.001:
            ax.text(j, i, '**', ha='center', va='center',
                    fontsize=12, fontweight='bold', color='black')
        elif len(row) > 0 and row.iloc[0]['FDR'] < 0.05:
            ax.text(j, i, '*', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='Signed -log₁₀(FDR)')
ax.set_title('Pathway Enrichment Across All Latent Components\n'
             'Red = Positive weight (↑risk) | Blue = Negative weight (↓protective)\n'
             '* FDR<0.05  ** FDR<0.001',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_heatmap_all_PCs.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("\n✓ Heatmap saved: fig_heatmap_all_PCs.png")

# =============================================================================
# 6. CORRELATION VALIDATION (all 10 PCs)
# =============================================================================
print(f"\n{'='*90}")
print("VALIDATION: Encoder Weight vs Expression Correlation (all PCs)")
print(f"{'='*90}")

ae_pca = pd.read_csv(r'C:\Users\nsw\Desktop\body_composition\ae_latent_pca10_all.csv')

g1 = pd.read_csv(r'C:\Users\nsw\Desktop\body_composition\RNA_Group1_updated_to_new_symbol.csv')
g2 = pd.read_csv(r'C:\Users\nsw\Desktop\body_composition\RNA_Group2_updated_to_new_symbol.csv')
g3 = pd.read_csv(r'C:\Users\nsw\Desktop\body_composition\RNA_Group3_updated_to_new_symbol.csv')
g4 = pd.read_csv(r'C:\Users\nsw\Desktop\body_composition\RNA_Group4_updated_to_new_symbol.csv')
rna = g1.set_index('gene')
for g in [g2, g3, g4]:
    rna = rna.join(g.set_index('gene'), how='outer')

common_samples = list(set(rna.columns) & set(ae_pca['sample_id']))
rna = rna[common_samples]
print(f"Samples: {len(common_samples)}")

for pc_idx in range(10):
    pc_name = f'PC{pc_idx+1}'
    ae_col = f'AE_{pc_name}'
    pc_vals = ae_pca.set_index('sample_id')[ae_col][common_samples].values
    pc_w = all_pc_weights[pc_name]

    corrs = []
    for gi, gene in enumerate(gene_names):
        if gene not in rna.index:
            continue
        expr = rna.loc[gene, common_samples].astype(float).values
        valid = ~(np.isnan(expr) | np.isnan(pc_vals))
        if valid.sum() > 50:
            r, _ = spearmanr(pc_vals[valid], expr[valid])
            corrs.append(r)
        else:
            corrs.append(np.nan)

    corr_arr = np.array(corrs)
    weight_arr = pc_w[:len(corrs)]
    valid_both = ~(np.isnan(corr_arr))
    rho, p = spearmanr(weight_arr[valid_both], corr_arr[valid_both])
    print(f"  {pc_name}: Weight vs Corr concordance rho={rho:.3f} (p={p:.1e})")

# =============================================================================
# 7. DETAILED FIGURES FOR PROGNOSTIC PC (PC6)
# =============================================================================
print(f"\n{'='*90}")
print("DETAILED FIGURES — PC6 (Prognostically Significant)")
print(f"{'='*90}")

pc6_w = weight_df['PC6']
pc6_vals = ae_pca.set_index('sample_id')['AE_PC6'][common_samples].values

# Correlation for PC6
corr_results = []
for gene in gene_names:
    if gene not in rna.index:
        continue
    expr = rna.loc[gene, common_samples].astype(float).values
    valid = ~(np.isnan(expr) | np.isnan(pc6_vals))
    if valid.sum() > 50:
        r, p = spearmanr(pc6_vals[valid], expr[valid])
        corr_results.append({'gene': gene, 'corr': r, 'p': p, 'weight': pc6_w[gene]})

corr_df = pd.DataFrame(corr_results)

# --- Figure A: Pathway enrichment bar (PC6 only) ---
pc6_enrich = enrich_df[enrich_df['PC'] == 'PC6'].sort_values('Mean_weight')
sig_pc6 = pc6_enrich[pc6_enrich['FDR'] < 0.1]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#1565C0' if w < 0 else '#D32F2F' for w in sig_pc6['Mean_weight']]
ax.barh(range(len(sig_pc6)), sig_pc6['Mean_weight'], color=colors, edgecolor='white')
ax.set_yticks(range(len(sig_pc6)))
ax.set_yticklabels(sig_pc6['Pathway'], fontsize=10)
for i, (_, r) in enumerate(sig_pc6.iterrows()):
    sig = '***' if r['FDR'] < 0.001 else '**' if r['FDR'] < 0.01 else '*' if r['FDR'] < 0.05 else '†'
    offset = 0.02 if r['Mean_weight'] > 0 else -0.02
    ha = 'left' if r['Mean_weight'] > 0 else 'right'
    ax.text(r['Mean_weight'] + offset, i, f"FDR={r['FDR']:.4f} {sig}",
            va='center', ha=ha, fontsize=8)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Mean Gene-to-PC6 Weight', fontsize=11, fontweight='bold')
ax.set_title('Pathway Enrichment — Prognostic Latent Feature\n'
             'Red = Risk (score↑) | Blue = Protective (score↓)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_pc6_pathway.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✓ PC6 pathway bar saved")

# --- Figure B: Volcano plot (PC6) ---
fig, ax = plt.subplots(figsize=(10, 6))
corr_df['neg_log10_p'] = -np.log10(corr_df['p'].clip(lower=1e-300))
ax.scatter(corr_df['weight'], corr_df['neg_log10_p'], alpha=0.3, s=15, color='#9E9E9E')

top_risk = corr_df.nlargest(10, 'weight')
top_prot = corr_df.nsmallest(10, 'weight')
ax.scatter(top_risk['weight'], top_risk['neg_log10_p'], color='#D32F2F', s=50, zorder=5, edgecolors='white')
ax.scatter(top_prot['weight'], top_prot['neg_log10_p'], color='#1565C0', s=50, zorder=5, edgecolors='white')

for _, r in pd.concat([top_risk.head(5), top_prot.head(5)]).iterrows():
    ax.annotate(r['gene'], (r['weight'], r['neg_log10_p']),
                fontsize=7, fontweight='bold', xytext=(5, 5), textcoords='offset points')

ax.axvline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('Gene-to-PC6 Encoder Weight', fontsize=11, fontweight='bold')
ax.set_ylabel('-log₁₀(p-value)', fontsize=11, fontweight='bold')
ax.set_title('Gene Contributions to Prognostic Latent Feature',
             fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_pc6_volcano.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✓ PC6 volcano saved")

# --- Figure C: Weight vs Correlation scatter (PC6) ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(corr_df['weight'], corr_df['corr'], alpha=0.2, s=15, color='#455A64')

pathway_colors = {
    'Glycolysis': '#D32F2F', 'Hypoxia': '#FF6F00',
    'Cholesterol Homeostasis': '#1565C0', 'mTORC1 Signaling': '#0D47A1',
}
for pathway, color in pathway_colors.items():
    genes_in = [g for g in hallmark_sets[pathway] if g in corr_df['gene'].values]
    sub = corr_df[corr_df['gene'].isin(genes_in)]
    ax.scatter(sub['weight'], sub['corr'], alpha=0.7, s=40, color=color,
               edgecolors='white', linewidth=0.5, label=pathway, zorder=5)

rho, p = spearmanr(corr_df['weight'], corr_df['corr'])
ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('Encoder Weight (Gene → PC)', fontsize=11, fontweight='bold')
ax.set_ylabel('Spearman Correlation (Gene ~ PC Score)', fontsize=11, fontweight='bold')
ax.set_title(f'Weight vs Correlation Concordance\nrho={rho:.3f} (p={p:.1e})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_pc6_weight_vs_corr.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✓ Weight vs correlation scatter saved")

print("\n✓ All done!")

Genes: 3000
Encoder: 3000 → 1024 → 256 → 64 → PCA(10)

✓ Gene weights for all 10 PCs saved: gene_weights_all_PCs.csv

TOP 5 RISK / PROTECTIVE GENES PER PC

  PC1:
    Risk↑:  KCTD6(+1.04), RAB40B(+0.88), RPAP1(+0.88), DNAH14(+0.84), REPIN1(+0.83)
    Prot↓:  GGT5(-0.96), MYL4(-0.96), HAPLN2(-0.94), SP140(-0.94), SLC15A3(-0.93)

  PC2:
    Risk↑:  SC5D(+1.20), DCTN6(+1.16), SARAF(+1.13), PNRC1(+1.07), SH3BGRL(+1.04)
    Prot↓:  MMP9(-1.33), NSUN5(-1.27), CLIC1(-1.26), RHBDF2(-1.23), COLGALT1(-1.21)

  PC3:
    Risk↑:  SH3TC1(+1.41), PFDN6(+1.30), TUSC2(+1.29), TMUB1(+1.23), FASTK(+1.22)
    Prot↓:  TOP2B(-1.32), ELK3(-1.29), SETD2(-1.29), CGAS(-1.28), HDAC9(-1.27)

  PC4:
    Risk↑:  GBP1(+1.40), PAK1IP1(+1.23), GPATCH4(+1.20), TAP2(+1.15), LYN(+1.14)
    Prot↓:  SH2B1(-1.48), HDAC11(-1.47), NEK8(-1.36), PPM1H(-1.22), RABEP2(-1.22)

  PC5:
    Risk↑:  CD96(+1.50), TAPBPL(+1.28), IKZF3(+1.26), PVRIG(+1.21), TNFRSF14(+1.19)
    Prot↓:  GFPT2(-1.52), LOXL2(-1.45), ITGA5(-1.40), COL6A2(-1.3